# AI B2B Supplier Selection Copilot — Clean Build

This notebook consolidates the working project from the uploaded notebook. It creates a **synthetic** supplier dataset, writes the current application/engine files, and runs repeatable smoke tests. It does not scrape IndiaMART or send real supplier enquiries.

**Run in Google Colab from top to bottom.** The Streamlit app is written to `app.py`; launch instructions are at the end.

In [1]:
# SETUP — run this cell first after a runtime restart
%pip -q install pandas numpy faker streamlit scikit-learn pytest openai
from pathlib import Path
import os, random, math, re, json, shutil, importlib, sys
from collections import defaultdict
import pandas as pd
import numpy as np
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker("en_IN")

OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", OUTPUT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.2 MB/s eta 0:00:00
Project directory: /content


## 1. Generate the synthetic dataset

In [2]:
# ============================================================
# PRODUCT CATALOGUE
# ============================================================

PRODUCT_CATALOG = {

    "Electrical & Lighting": {

        "Industrial LED Bulb": [
            {
                "variant": "Industrial LED Bulb 20W",
                "price_range": (160, 280),
                "price_unit": "per piece",
                "moq_range": (50, 500),
                "warranty_range": (12, 24),
                "attributes": {
                    "wattage_w": 20,
                    "light_color": ["Cool White", "Daylight"],
                    "cct_k": [4000, 6500],
                    "ip_rating": ["IP20", "IP44"],
                    "lumens": (1800, 2200)
                }
            },

            {
                "variant": "Industrial LED Bulb 30W",
                "price_range": (240, 420),
                "price_unit": "per piece",
                "moq_range": (50, 500),
                "warranty_range": (12, 36),
                "attributes": {
                    "wattage_w": 30,
                    "light_color": ["Cool White", "Daylight"],
                    "cct_k": [4000, 6500],
                    "ip_rating": ["IP20", "IP44"],
                    "lumens": (2700, 3300)
                }
            },

            {
                "variant": "Industrial LED Bulb 50W",
                "price_range": (400, 700),
                "price_unit": "per piece",
                "moq_range": (25, 250),
                "warranty_range": (12, 36),
                "attributes": {
                    "wattage_w": 50,
                    "light_color": ["Cool White", "Daylight"],
                    "cct_k": [4000, 6500],
                    "ip_rating": ["IP44", "IP65"],
                    "lumens": (4500, 5500)
                }
            }
        ],

        "LED Flood Light": [
            {
                "variant": "LED Flood Light 50W IP65",
                "price_range": (500, 1000),
                "price_unit": "per piece",
                "moq_range": (10, 100),
                "warranty_range": (12, 36),
                "attributes": {
                    "wattage_w": 50,
                    "ip_rating": ["IP65"],
                    "lumens": (4500, 6000)
                }
            },

            {
                "variant": "LED Flood Light 100W IP65",
                "price_range": (950, 1800),
                "price_unit": "per piece",
                "moq_range": (10, 100),
                "warranty_range": (12, 36),
                "attributes": {
                    "wattage_w": 100,
                    "ip_rating": ["IP65"],
                    "lumens": (9000, 12000)
                }
            }
        ],

        "Copper Wire": [
            {
                "variant": "FR Copper Wire 1.5 sq mm 90m",
                "price_range": (1400, 2200),
                "price_unit": "per coil (90m)",
                "moq_range": (5, 50),
                "warranty_range": (12, 60),
                "attributes": {
                    "cross_section_sqmm": 1.5,
                    "conductor": ["Copper"],
                    "insulation": ["FR", "FRLS"]
                }
            },

            {
                "variant": "FR Copper Wire 2.5 sq mm 90m",
                "price_range": (2300, 3600),
                "price_unit": "per coil (90m)",
                "moq_range": (5, 50),
                "warranty_range": (12, 60),
                "attributes": {
                    "cross_section_sqmm": 2.5,
                    "conductor": ["Copper"],
                    "insulation": ["FR", "FRLS"]
                }
            }
        ],

        "MCB": [
            {
                "variant": "MCB SP C-Curve 16A 10kA",
                "price_range": (100, 220),
                "price_unit": "per piece",
                "moq_range": (20, 100),
                "warranty_range": (12, 36),
                "attributes": {
                    "poles": [1],
                    "current_a": [16],
                    "curve": ["C"],
                    "breaking_capacity_ka": [10]
                }
            },

            {
                "variant": "MCB DP C-Curve 32A 10kA",
                "price_range": (300, 650),
                "price_unit": "per piece",
                "moq_range": (20, 100),
                "warranty_range": (12, 36),
                "attributes": {
                    "poles": [2],
                    "current_a": [32],
                    "curve": ["C"],
                    "breaking_capacity_ka": [10]
                }
            }
        ]
    },


    # --------------------------------------------------------
    # OFFICE FURNITURE
    # --------------------------------------------------------

    "Office Furniture": {

        "Ergonomic Chair": [
            {
                "variant": "Ergonomic Chair Mid-Back Mesh",
                "price_range": (3500, 6000),
                "price_unit": "per piece",
                "moq_range": (5, 30),
                "warranty_range": (12, 60),
                "attributes": {
                    "weight_capacity_kg": (100, 120),
                    "armrest": ["Fixed", "2D"],
                    "lumbar_support": ["Adjustable"]
                }
            },

            {
                "variant": "Ergonomic Chair High-Back Mesh",
                "price_range": (5500, 9500),
                "price_unit": "per piece",
                "moq_range": (5, 30),
                "warranty_range": (24, 60),
                "attributes": {
                    "weight_capacity_kg": (120, 150),
                    "armrest": ["2D", "3D"],
                    "lumbar_support": ["Adjustable"]
                }
            }
        ],

        "Office Desk": [
            {
                "variant": "Office Desk 4ft",
                "price_range": (4000, 7000),
                "price_unit": "per piece",
                "moq_range": (5, 20),
                "warranty_range": (12, 36),
                "attributes": {
                    "length_ft": [4],
                    "top_material": ["MDF", "Engineered Wood"]
                }
            },

            {
                "variant": "Office Desk 6ft",
                "price_range": (8500, 16000),
                "price_unit": "per piece",
                "moq_range": (5, 20),
                "warranty_range": (12, 36),
                "attributes": {
                    "length_ft": [6],
                    "top_material": ["MDF", "Engineered Wood"]
                }
            }
        ]
    },


    # --------------------------------------------------------
    # METALS
    # --------------------------------------------------------

    "Metals & Raw Materials": {

        "TMT Steel": [
            {
                "variant": "TMT Fe500D 8mm",
                "price_range": (55000, 68000),
                "price_unit": "per tonne",
                "moq_range": (2, 20),
                "warranty_range": None,
                "attributes": {
                    "grade": ["Fe500D"],
                    "diameter_mm": [8]
                }
            },

            {
                "variant": "TMT Fe500D 12mm",
                "price_range": (54000, 67000),
                "price_unit": "per tonne",
                "moq_range": (2, 20),
                "warranty_range": None,
                "attributes": {
                    "grade": ["Fe500D"],
                    "diameter_mm": [12]
                }
            },

            {
                "variant": "TMT Fe500D 16mm",
                "price_range": (53500, 66500),
                "price_unit": "per tonne",
                "moq_range": (2, 20),
                "warranty_range": None,
                "attributes": {
                    "grade": ["Fe500D"],
                    "diameter_mm": [16]
                }
            }
        ],

        "MS Steel Sheet": [
            {
                "variant": "MS Sheet IS 2062 3mm",
                "price_range": (55, 75),
                "price_unit": "per kg",
                "moq_range": (500, 3000),
                "warranty_range": None,
                "attributes": {
                    "grade": ["IS 2062"],
                    "thickness_mm": [3]
                }
            },

            {
                "variant": "MS Sheet IS 2062 5mm",
                "price_range": (55, 75),
                "price_unit": "per kg",
                "moq_range": (500, 3000),
                "warranty_range": None,
                "attributes": {
                    "grade": ["IS 2062"],
                    "thickness_mm": [5]
                }
            }
        ]
    },


    # --------------------------------------------------------
    # PACKAGING
    # --------------------------------------------------------

    "Packaging Materials": {

        "Corrugated Box": [
            {
                "variant": "Corrugated Box 3-Ply 12x10x8 inch",
                "price_range": (12, 24),
                "price_unit": "per piece",
                "moq_range": (500, 3000),
                "warranty_range": None,
                "attributes": {
                    "ply": [3],
                    "printing": ["Plain", "1-colour"],
                    "load_capacity_kg": (8, 15)
                }
            },

            {
                "variant": "Corrugated Box 5-Ply 18x12x12 inch",
                "price_range": (35, 65),
                "price_unit": "per piece",
                "moq_range": (500, 3000),
                "warranty_range": None,
                "attributes": {
                    "ply": [5],
                    "printing": ["Plain", "1-colour", "2-colour"],
                    "load_capacity_kg": (15, 30)
                }
            },

            {
                "variant": "Corrugated Box 5-Ply 24x16x16 inch",
                "price_range": (60, 115),
                "price_unit": "per piece",
                "moq_range": (500, 3000),
                "warranty_range": None,
                "attributes": {
                    "ply": [5],
                    "printing": ["Plain", "1-colour", "2-colour"],
                    "load_capacity_kg": (20, 40)
                }
            }
        ],

        "BOPP Tape": [
            {
                "variant": "BOPP Tape 48mm x 65m",
                "price_range": (25, 50),
                "price_unit": "per roll",
                "moq_range": (36, 144),
                "warranty_range": None,
                "attributes": {
                    "width_mm": [48],
                    "length_m": [65]
                }
            },

            {
                "variant": "BOPP Tape 48mm x 100m",
                "price_range": (38, 70),
                "price_unit": "per roll",
                "moq_range": (36, 144),
                "warranty_range": None,
                "attributes": {
                    "width_mm": [48],
                    "length_m": [100]
                }
            }
        ]
    },


    # --------------------------------------------------------
    # INDUSTRIAL SAFETY
    # --------------------------------------------------------

    "Industrial Safety Equipment": {

        "Safety Helmet": [
            {
                "variant": "Safety Helmet HDPE Ratchet",
                "price_range": (130, 240),
                "price_unit": "per piece",
                "moq_range": (50, 250),
                "warranty_range": (0, 12),
                "attributes": {
                    "material": ["HDPE"],
                    "suspension": ["6-point"],
                    "chin_strap": [True, False]
                }
            },

            {
                "variant": "Safety Helmet ABS Vented",
                "price_range": (200, 400),
                "price_unit": "per piece",
                "moq_range": (50, 250),
                "warranty_range": (0, 12),
                "attributes": {
                    "material": ["ABS"],
                    "suspension": ["6-point"],
                    "chin_strap": [True]
                }
            }
        ],

        "Safety Shoes": [
            {
                "variant": "Safety Shoes Steel Toe Low Ankle",
                "price_range": (500, 900),
                "price_unit": "per pair",
                "moq_range": (24, 100),
                "warranty_range": (3, 12),
                "attributes": {
                    "toe_cap": ["Steel"],
                    "ankle": ["Low"]
                }
            },

            {
                "variant": "Safety Shoes Steel Toe High Ankle",
                "price_range": (700, 1300),
                "price_unit": "per pair",
                "moq_range": (24, 100),
                "warranty_range": (3, 12),
                "attributes": {
                    "toe_cap": ["Steel"],
                    "ankle": ["High"]
                }
            }
        ]
    }
}

print("Categories:", len(PRODUCT_CATALOG))

Categories: 5


In [3]:
import pandas as pd

# ============================================================
# FLATTEN PRODUCT CATALOGUE
# ============================================================

products = []

for category, families in PRODUCT_CATALOG.items():

    for family, variants in families.items():

        for item in variants:

            row = item.copy()

            row["category"] = category
            row["product_family"] = family

            products.append(row)


product_df = pd.DataFrame(products)

print("Unique product variants:", len(product_df))

display(
    product_df[
        [
            "category",
            "product_family",
            "variant",
            "price_range",
            "price_unit"
        ]
    ]
)

Unique product variants: 27


,category,product_family,variant,price_range,price_unit
0,Electrical & Lighting,Industrial LED Bulb,Industrial LED Bulb 20W,"(160, 280)",per piece
1,Electrical & Lighting,Industrial LED Bulb,Industrial LED Bulb 30W,"(240, 420)",per piece
2,Electrical & Lighting,Industrial LED Bulb,Industrial LED Bulb 50W,"(400, 700)",per piece
3,Electrical & Lighting,LED Flood Light,LED Flood Light 50W IP65,"(500, 1000)",per piece
4,Electrical & Lighting,LED Flood Light,LED Flood Light 100W IP65,"(950, 1800)",per piece
5,Electrical & Lighting,Copper Wire,FR Copper Wire 1.5 sq mm 90m,"(1400, 2200)",per coil (90m)
6,Electrical & Lighting,Copper Wire,FR Copper Wire 2.5 sq mm 90m,"(2300, 3600)",per coil (90m)
7,Electrical & Lighting,MCB,MCB SP C-Curve 16A 10kA,"(100, 220)",per piece
8,Electrical & Lighting,MCB,MCB DP C-Curve 32A 10kA,"(300, 650)",per piece
9,Office Furniture,Ergonomic Chair,Ergonomic Chair Mid-Back Mesh,"(3500, 6000)",per piece


In [4]:
# ============================================================
# INDIAN BUSINESS LOCATIONS
# ============================================================

CITY_MAP = {

    "Maharashtra":
        ["Mumbai", "Pune", "Nashik", "Nagpur", "Bhiwandi"],

    "Gujarat":
        ["Ahmedabad", "Vadodara", "Surat", "Rajkot", "Vapi"],

    "Karnataka":
        ["Bengaluru", "Mysuru", "Hubballi"],

    "Tamil Nadu":
        ["Chennai", "Coimbatore", "Hosur"],

    "Telangana":
        ["Hyderabad", "Warangal"],

    "Delhi":
        ["New Delhi"],

    "Haryana":
        ["Gurugram", "Faridabad", "Panipat"],

    "Uttar Pradesh":
        ["Noida", "Ghaziabad", "Kanpur"],

    "Rajasthan":
        ["Jaipur", "Udaipur"],

    "West Bengal":
        ["Kolkata", "Durgapur"],

    "Odisha":
        ["Bhubaneswar", "Cuttack", "Rourkela"],

    "Chhattisgarh":
        ["Raipur", "Bhilai"],

    "Jharkhand":
        ["Jamshedpur", "Ranchi"],

    "Punjab":
        ["Ludhiana", "Mohali"]
}

states = list(CITY_MAP.keys())

print("States:", len(states))

States: 14


In [5]:
# ============================================================
# SUPPLIER TYPES
# ============================================================

SUPPLIER_TYPES = {

    "Premium Manufacturer": {
        "price_multiplier": 1.08,
        "quality": 0.90,
        "response": 0.94,
        "on_time": 0.95
    },

    "Balanced Manufacturer": {
        "price_multiplier": 1.00,
        "quality": 0.78,
        "response": 0.88,
        "on_time": 0.89
    },

    "Regional Distributor": {
        "price_multiplier": 1.02,
        "quality": 0.70,
        "response": 0.92,
        "on_time": 0.94
    },

    "Volume Wholesaler": {
        "price_multiplier": 0.94,
        "quality": 0.63,
        "response": 0.80,
        "on_time": 0.82
    },

    "Budget Supplier": {
        "price_multiplier": 0.88,
        "quality": 0.52,
        "response": 0.70,
        "on_time": 0.75
    }
}

In [6]:
# Imports required by this cell (safe even if the setup cell was skipped)
import random
import pandas as pd
import numpy as np

# ============================================================
# SUPPLIER MASTER
# ============================================================

prefixes = [
    "Apex",
    "Prime",
    "Vertex",
    "Nova",
    "Trinity",
    "Shree",
    "Metro",
    "Eastern",
    "Western",
    "United",
    "Precision",
    "Supreme",
    "Dynamic",
    "Sterling",
    "National"
]

suffixes = [
    "Industries",
    "Enterprises",
    "Engineering",
    "Solutions",
    "Traders",
    "Distributors",
    "Industrial Works",
    "Supplies"
]

suppliers = []

for i in range(1, 201):

    supplier_type = random.choice(
        list(SUPPLIER_TYPES.keys())
    )

    state = random.choice(states)

    city = random.choice(
        CITY_MAP[state]
    )

    profile = SUPPLIER_TYPES[supplier_type]

    quality = np.clip(
        np.random.normal(
            profile["quality"],
            0.06
        ),
        0.35,
        0.98
    )

    rating = np.clip(
        3.0 + quality * 2.0 +
        np.random.normal(0, 0.08),
        3.0,
        4.9
    )

    response = np.clip(
        profile["response"] * 100 +
        np.random.normal(0, 3),
        55,
        99
    )

    on_time = np.clip(
        profile["on_time"] * 100 +
        np.random.normal(0, 3),
        60,
        99
    )

    years = int(
        np.clip(
            np.random.gamma(3.5, 2.5),
            1,
            30
        )
    )

    suppliers.append({

        "supplier_id":
            f"S{i:04d}",

        "supplier_name":
            f"{random.choice(prefixes)} "
            f"{random.choice(suffixes)} {i}",

        "supplier_type":
            supplier_type,

        "state":
            state,

        "city":
            city,

        "years_in_business":
            years,

        "rating":
            round(rating, 2),

        "response_rate_pct":
            round(response, 1),

        "on_time_delivery_pct":
            round(on_time, 1),

        "verified_supplier":
            random.random() < 0.94
    })


supplier_df = pd.DataFrame(suppliers)

display(supplier_df.head())

print(
    "Suppliers:",
    len(supplier_df)
)

,supplier_id,supplier_name,supplier_type,state,city,years_in_business,rating,response_rate_pct,on_time_delivery_pct,verified_supplier
0,S0001,Nova Solutions 1,Premium Manufacturer,Maharashtra,Nashik,6,4.85,95.9,99.0,True
1,S0002,National Enterprises 2,Premium Manufacturer,Odisha,Rourkela,5,4.79,97.0,93.3,True
2,S0003,Nova Solutions 3,Premium Manufacturer,Maharashtra,Mumbai,4,4.68,88.8,93.3,True
3,S0004,Supreme Industrial Works 4,Premium Manufacturer,Rajasthan,Jaipur,10,4.78,87.6,93.1,True
4,S0005,Dynamic Engineering 5,Budget Supplier,Telangana,Hyderabad,5,3.98,66.5,76.1,True


Suppliers: 200


In [7]:
# Imports required by this cell (safe even if the setup cell was skipped)
import random
import pandas as pd
import numpy as np

# ============================================================
# GENERATE SUPPLIER LISTINGS
# ============================================================

listings = []

listing_id = 1


for _, supplier in supplier_df.iterrows():

    # Every supplier sells 3–7 variants
    number_of_products = random.randint(3, 7)

    selected_products = product_df.sample(
        number_of_products,
        random_state=random.randint(1, 100000)
    )

    for _, product in selected_products.iterrows():

        base_low, base_high = product["price_range"]

        supplier_type = supplier["supplier_type"]

        multiplier = (
            SUPPLIER_TYPES[supplier_type]
            ["price_multiplier"]
        )

        # Natural price variation
        price = np.random.uniform(
            base_low,
            base_high
        ) * multiplier

        # Small supplier-specific variation
        price *= np.random.normal(
            1.0,
            0.035
        )

        price = max(
            base_low * 0.80,
            min(
                base_high * 1.20,
                price
            )
        )

        price = round(price, 2)

        # ----------------------------------------------------
        # MOQ
        # ----------------------------------------------------

        moq_low, moq_high = product["moq_range"]

        moq = random.randint(
            moq_low,
            moq_high
        )

        # ----------------------------------------------------
        # INVENTORY
        # ----------------------------------------------------

        inventory_multiplier = random.uniform(
            3,
            20
        )

        available_quantity = int(
            moq * inventory_multiplier
        )

        # ----------------------------------------------------
        # DELIVERY
        # ----------------------------------------------------

        if supplier_type == "Regional Distributor":
            dispatch_days = random.randint(1, 4)

        elif supplier_type == "Premium Manufacturer":
            dispatch_days = random.randint(2, 6)

        elif supplier_type == "Balanced Manufacturer":
            dispatch_days = random.randint(2, 7)

        elif supplier_type == "Volume Wholesaler":
            dispatch_days = random.randint(2, 6)

        else:
            dispatch_days = random.randint(3, 10)

        # ----------------------------------------------------
        # WARRANTY
        # ----------------------------------------------------

        warranty_range = product["warranty_range"]

        if warranty_range is None:

            warranty_months = None

        else:

            w_low, w_high = warranty_range

            quality_factor = (
                supplier["rating"] - 3
            ) / 2

            warranty = (
                w_low +
                (w_high - w_low)
                * quality_factor
                * random.uniform(
                    0.7,
                    1.0
                )
            )

            warranty_months = int(
                np.clip(
                    warranty,
                    w_low,
                    w_high
                )
            )

        # ----------------------------------------------------
        # BULK DISCOUNT
        # ----------------------------------------------------

        if supplier_type == "Volume Wholesaler":
            discount_1 = random.uniform(3, 7)
            discount_2 = random.uniform(6, 12)

        elif supplier_type == "Premium Manufacturer":
            discount_1 = random.uniform(1, 4)
            discount_2 = random.uniform(3, 7)

        else:
            discount_1 = random.uniform(1, 5)
            discount_2 = random.uniform(3, 9)

        # ----------------------------------------------------
        # PRODUCT ATTRIBUTES
        # ----------------------------------------------------

        attributes = {}

        for attr, values in product["attributes"].items():

            if isinstance(values, tuple):

                if all(
                    isinstance(x, (int, float))
                    for x in values
                ):

                    attributes[attr] = round(
                        random.uniform(
                            values[0],
                            values[1]
                        ),
                        2
                    )

            elif isinstance(values, list):

                attributes[attr] = random.choice(
                    values
                )

        # ----------------------------------------------------
        # DESCRIPTION
        # ----------------------------------------------------

        description = (
            f"{product['variant']} supplied by "
            f"{supplier['supplier_name']} "
            f"for B2B procurement."
        )

        listings.append({

            "listing_id":
                f"L{listing_id:05d}",

            "supplier_id":
                supplier["supplier_id"],

            "supplier_name":
                supplier["supplier_name"],

            "category":
                product["category"],

            "product_family":
                product["product_family"],

            "variant":
                product["variant"],

            # IMPORTANT:
            # PRICE IS FOR ONE UNIT
            "price_inr":
                price,

            "price_unit":
                product["price_unit"],

            "moq":
                moq,

            "available_quantity":
                available_quantity,

            "dispatch_days":
                dispatch_days,

            "warranty_months":
                warranty_months,

            "rating":
                supplier["rating"],

            "response_rate_pct":
                supplier[
                    "response_rate_pct"
                ],

            "on_time_delivery_pct":
                supplier[
                    "on_time_delivery_pct"
                ],

            "verified_supplier":
                supplier[
                    "verified_supplier"
                ],

            "bulk_discount_1_pct":
                round(
                    discount_1,
                    2
                ),

            "bulk_discount_2_pct":
                round(
                    discount_2,
                    2
                ),

            "description":
                description,

            **attributes
        })

        listing_id += 1


listing_df = pd.DataFrame(listings)

print(
    "Listings generated:",
    len(listing_df)
)

print(
    "Unique suppliers:",
    listing_df["supplier_id"].nunique()
)

print(
    "Unique variants:",
    listing_df["variant"].nunique()
)

display(listing_df.head())

Listings generated: 1020
Unique suppliers: 200
Unique variants: 27


,listing_id,supplier_id,supplier_name,category,product_family,variant,price_inr,price_unit,moq,available_quantity,...,material,suspension,chin_strap,toe_cap,ankle,grade,thickness_mm,width_mm,length_m,diameter_mm
0,L00001,S0001,Nova Solutions 1,Electrical & Lighting,Industrial LED Bulb,Industrial LED Bulb 30W,402.30,per piece,237,1074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L00002,S0001,Nova Solutions 1,Electrical & Lighting,MCB,MCB SP C-Curve 16A 10kA,212.39,per piece,49,539,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,L00003,S0001,Nova Solutions 1,Office Furniture,Ergonomic Chair,Ergonomic Chair High-Back Mesh,9368.50,per piece,25,248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,L00004,S0001,Nova Solutions 1,Office Furniture,Office Desk,Office Desk 4ft,7424.09,per piece,17,182,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,L00005,S0001,Nova Solutions 1,Packaging Materials,Corrugated Box,Corrugated Box 3-Ply 12x10x8 inch,13.65,per piece,2504,33719,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ============================================================
# UNIT PRICE / ORDER COST FUNCTIONS
# ============================================================

def calculate_order_cost(row, quantity):

    # Supplier cannot fulfil order
    if quantity < row["moq"]:
        return None

    if quantity > row["available_quantity"]:
        return None

    unit_price = row["price_inr"]

    # Bulk discount
    if quantity >= row["moq"] * 5:

        discount = (
            row["bulk_discount_2_pct"]
        )

    elif quantity >= row["moq"] * 2:

        discount = (
            row["bulk_discount_1_pct"]
        )

    else:

        discount = 0

    effective_unit_price = (
        unit_price *
        (1 - discount / 100)
    )

    total_cost = (
        effective_unit_price *
        quantity
    )

    return {
        "base_unit_price": unit_price,
        "discount_pct": discount,
        "effective_unit_price":
            round(
                effective_unit_price,
                2
            ),
        "quantity":
            quantity,
        "estimated_total_cost":
            round(
                total_cost,
                2
            )
    }

In [9]:
# ============================================================
# PRICE/UNIT SANITY CHECK
# ============================================================

sample = listing_df.sample(10)

for _, row in sample.iterrows():

    print(
        f"{row['variant']} | "
        f"₹{row['price_inr']} "
        f"{row['price_unit']} | "
        f"MOQ: {row['moq']} | "
        f"Stock: {row['available_quantity']}"
    )

Safety Shoes Steel Toe High Ankle | ₹945.28 per pair | MOQ: 98 | Stock: 1956
TMT Fe500D 8mm | ₹55058.63 per tonne | MOQ: 3 | Stock: 17
LED Flood Light 50W IP65 | ₹704.42 per piece | MOQ: 99 | Stock: 404
Office Desk 4ft | ₹4168.97 per piece | MOQ: 5 | Stock: 55
Corrugated Box 5-Ply 18x12x12 inch | ₹59.19 per piece | MOQ: 2631 | Stock: 34566
FR Copper Wire 2.5 sq mm 90m | ₹2111.52 per coil (90m) | MOQ: 31 | Stock: 192
Ergonomic Chair High-Back Mesh | ₹7755.87 per piece | MOQ: 27 | Stock: 176
TMT Fe500D 12mm | ₹56660.66 per tonne | MOQ: 9 | Stock: 109
Industrial LED Bulb 20W | ₹251.24 per piece | MOQ: 101 | Stock: 1012
Safety Shoes Steel Toe High Ankle | ₹994.7 per pair | MOQ: 24 | Stock: 120


In [10]:
# ============================================================
# DATASET VALIDATION
# ============================================================

errors = []

# ------------------------------------------------------------
# Price
# ------------------------------------------------------------

if (listing_df["price_inr"] <= 0).any():
    errors.append(
        "Found non-positive prices"
    )


# ------------------------------------------------------------
# MOQ
# ------------------------------------------------------------

if (
    listing_df["moq"] <= 0
).any():

    errors.append(
        "Found invalid MOQ"
    )


# ------------------------------------------------------------
# Inventory
# ------------------------------------------------------------

if (
    listing_df["available_quantity"]
    < listing_df["moq"]
).any():

    errors.append(
        "Inventory below MOQ"
    )


# ------------------------------------------------------------
# Rating
# ------------------------------------------------------------

if (
    ~listing_df["rating"].between(
        3,
        5
    )
).any():

    errors.append(
        "Invalid rating"
    )


# ------------------------------------------------------------
# Duplicate listing IDs
# ------------------------------------------------------------

if (
    listing_df["listing_id"]
    .duplicated()
    .any()
):

    errors.append(
        "Duplicate listing ID"
    )


# ------------------------------------------------------------
# Supplier + variant duplicate
# ------------------------------------------------------------

if (
    listing_df
    .duplicated(
        ["supplier_id", "variant"]
    )
    .any()
):

    errors.append(
        "Duplicate supplier-variant"
    )


# ------------------------------------------------------------
# Price sanity against catalogue
# ------------------------------------------------------------

for _, row in listing_df.iterrows():

    product = product_df[
        product_df["variant"]
        == row["variant"]
    ].iloc[0]

    low, high = product[
        "price_range"
    ]

    if not (
        low * 0.75
        <= row["price_inr"]
        <= high * 1.25
    ):

        errors.append(
            f"Unrealistic price: "
            f"{row['variant']} "
            f"₹{row['price_inr']}"
        )


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

if len(errors) == 0:

    print(
        "✅ ALL VALIDATION CHECKS PASSED"
    )

else:

    print(
        "❌ ERRORS FOUND:"
    )

    for error in errors:
        print(
            "-",
            error
        )

DATASET VALIDATION
✅ ALL VALIDATION CHECKS PASSED


In [11]:
# ============================================================
# SUPPLIER COMPETITION
# ============================================================

competition = (

    listing_df

    .groupby(
        [
            "category",
            "product_family",
            "variant"
        ]
    )

    .agg(
        suppliers=(
            "supplier_id",
            "nunique"
        ),

        min_price=(
            "price_inr",
            "min"
        ),

        max_price=(
            "price_inr",
            "max"
        ),

        avg_price=(
            "price_inr",
            "mean"
        )
    )

    .reset_index()
)

display(
    competition.sort_values(
        "suppliers"
    )
)

,category,product_family,variant,suppliers,min_price,max_price,avg_price
8,Electrical & Lighting,MCB,MCB SP C-Curve 16A 10kA,30,101.18,250.91,157.160333
12,Industrial Safety Equipment,Safety Shoes,Safety Shoes Steel Toe Low Ankle,32,442.20,890.68,690.014688
19,Office Furniture,Ergonomic Chair,Ergonomic Chair Mid-Back Mesh,32,3552.07,6407.11,4803.834375
4,Electrical & Lighting,Industrial LED Bulb,Industrial LED Bulb 50W,33,411.02,769.98,574.333333
25,Packaging Materials,Corrugated Box,Corrugated Box 5-Ply 18x12x12 inch,33,32.74,66.90,49.354242
22,Packaging Materials,BOPP Tape,BOPP Tape 48mm x 100m,34,38.62,68.75,54.534412
7,Electrical & Lighting,MCB,MCB DP C-Curve 32A 10kA,34,284.69,683.65,460.282353
5,Electrical & Lighting,LED Flood Light,LED Flood Light 100W IP65,34,922.05,1892.55,1397.411765
23,Packaging Materials,BOPP Tape,BOPP Tape 48mm x 65m,34,23.96,51.46,38.060000
17,Metals & Raw Materials,TMT Steel,TMT Fe500D 8mm,35,49881.76,76150.34,59874.662571


In [12]:
# ============================================================
# SAVE DATASET
# ============================================================
from pathlib import Path

# Defensive fallback: OUTPUT_DIR is normally set in the first setup cell.
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path.cwd()
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in ("supplier_df", "listing_df") if name not in globals()]
if missing:
    raise RuntimeError(
        "Dataset variables are missing: " + ", ".join(missing) + ". "
        "Run the notebook cells from the top in order, especially the Supplier Master "
        "and Generate Supplier Listings cells, then rerun this cell."
    )

product_path = OUTPUT_DIR / "product_catalogue.csv"
supplier_path = OUTPUT_DIR / "supplier_master.csv"
listing_path = OUTPUT_DIR / "supplier_listings.csv"

# The Streamlit app loads all three files, so save all three datasets.
if "product_df" not in globals():
    raise RuntimeError(
        "product_df is missing. Run the Product Catalogue and Flatten Product Catalogue cells first."
    )

product_df.to_csv(product_path, index=False)
supplier_df.to_csv(supplier_path, index=False)
listing_df.to_csv(listing_path, index=False)

print("Product catalogue:", product_path)
print("Supplier master:", supplier_path)
print("Supplier listings:", listing_path)
print("Rows saved — products:", len(product_df), "| suppliers:", len(supplier_df), "| listings:", len(listing_df))


Product catalogue: /content/product_catalogue.csv
Supplier master: /content/supplier_master.csv
Supplier listings: /content/supplier_listings.csv
Rows saved — products: 27 | suppliers: 200 | listings: 1020


## 2. Create location master and complete transit assumptions

In [13]:
from pathlib import Path
Path("india_locations.py").write_text('# ============================================================\n# INDIA BUYER LOCATION MASTER\n# 28 STATES + 8 UNION TERRITORIES\n# ============================================================\n\nINDIA_LOCATIONS = [\n    # States\n    "Andhra Pradesh",\n    "Arunachal Pradesh",\n    "Assam",\n    "Bihar",\n    "Chhattisgarh",\n    "Goa",\n    "Gujarat",\n    "Haryana",\n    "Himachal Pradesh",\n    "Jharkhand",\n    "Karnataka",\n    "Kerala",\n    "Madhya Pradesh",\n    "Maharashtra",\n    "Manipur",\n    "Meghalaya",\n    "Mizoram",\n    "Nagaland",\n    "Odisha",\n    "Punjab",\n    "Rajasthan",\n    "Sikkim",\n    "Tamil Nadu",\n    "Telangana",\n    "Tripura",\n    "Uttar Pradesh",\n    "Uttarakhand",\n    "West Bengal",\n\n    # Union Territories\n    "Andaman and Nicobar Islands",\n    "Chandigarh",\n    "Dadra and Nagar Haveli and Daman and Diu",\n    "Delhi",\n    "Jammu and Kashmir",\n    "Ladakh",\n    "Lakshadweep",\n    "Puducherry"\n]\n\n\n# ============================================================\n# VALIDATION\n# ============================================================\n\nassert len(INDIA_LOCATIONS) == 36\nassert len(set(INDIA_LOCATIONS)) == 36\n\nprint("India location master created successfully.")\nprint(f"Total States + UTs: {len(INDIA_LOCATIONS)}")\n\nprint("\\nStates + UTs:")\nfor i, location in enumerate(INDIA_LOCATIONS, start=1):\n    print(f"{i:02d}. {location}")' , encoding="utf-8")
exec(compile(Path("india_locations.py").read_text(encoding="utf-8"), "india_locations.py", "exec"))
print("Wrote india_locations.py")

India location master created successfully.
Total States + UTs: 36

States + UTs:
01. Andhra Pradesh
02. Arunachal Pradesh
03. Assam
04. Bihar
05. Chhattisgarh
06. Goa
07. Gujarat
08. Haryana
09. Himachal Pradesh
10. Jharkhand
11. Karnataka
12. Kerala
13. Madhya Pradesh
14. Maharashtra
15. Manipur
16. Meghalaya
17. Mizoram
18. Nagaland
19. Odisha
20. Punjab
21. Rajasthan
22. Sikkim
23. Tamil Nadu
24. Telangana
25. Tripura
26. Uttar Pradesh
27. Uttarakhand
28. West Bengal
29. Andaman and Nicobar Islands
30. Chandigarh
31. Dadra and Nagar Haveli and Daman and Diu
32. Delhi
33. Jammu and Kashmir
34. Ladakh
35. Lakshadweep
36. Puducherry
Wrote india_locations.py


## 3. Write the consolidated supplier-selection engine

In [14]:
from pathlib import Path
ml_risk_source = '"""Unsupervised ML risk screening for synthetic supplier profiles.\n\nThis is an anomaly-detection signal, not a prediction of fraud or future delivery.\nBecause the data is synthetic and has no verified outcome labels, the model is\nnot presented as a supervised performance predictor.\n"""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import IsolationForest\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.pipeline import make_pipeline\n\nFEATURES = [\n    "years_in_business",\n    "rating",\n    "response_rate_pct",\n    "on_time_delivery_pct",\n    "verified_numeric",\n    "listing_count",\n    "mean_price_cv_pct",\n]\n\n\ndef build_supplier_features(supplier_df, listing_df=None):\n    """Build one numeric feature row per supplier using available profile data."""\n    frame = supplier_df.copy()\n    frame["verified_numeric"] = frame["verified_supplier"].astype(bool).astype(int)\n    if listing_df is not None and not listing_df.empty:\n        agg = listing_df.groupby("supplier_id").agg(\n            listing_count=("listing_id", "count"),\n            mean_listing_price=("price_inr", "mean"),\n            std_listing_price=("price_inr", "std"),\n        ).reset_index()\n        agg["mean_price_cv_pct"] = (\n            agg["std_listing_price"].fillna(0) / agg["mean_listing_price"].replace(0, np.nan) * 100\n        ).replace([np.inf, -np.inf], np.nan).fillna(0)\n        frame = frame.merge(agg[["supplier_id", "listing_count", "mean_price_cv_pct"]], on="supplier_id", how="left")\n    else:\n        frame["listing_count"] = 0\n        frame["mean_price_cv_pct"] = 0.0\n    frame["listing_count"] = frame["listing_count"].fillna(0)\n    frame["mean_price_cv_pct"] = frame["mean_price_cv_pct"].fillna(0)\n    for col in FEATURES:\n        if col not in frame:\n            frame[col] = 0\n        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)\n        frame[col] = frame[col].fillna(frame[col].median() if frame[col].notna().any() else 0)\n    return frame[["supplier_id"] + FEATURES]\n\n\ndef score_supplier_anomalies(supplier_df, listing_df=None, contamination=0.08, random_state=42):\n    """Fit Isolation Forest and return a 0-100 relative anomaly score per supplier."""\n    features = build_supplier_features(supplier_df, listing_df)\n    if len(features) < 10:\n        return pd.DataFrame({\n            "supplier_id": features["supplier_id"],\n            "ml_anomaly_score": 0.0,\n            "ml_risk_level": "Insufficient data",\n            "ml_risk_note": "At least 10 supplier profiles are needed for anomaly screening.",\n        })\n    model = make_pipeline(\n        StandardScaler(),\n        IsolationForest(contamination=contamination, random_state=random_state, n_estimators=200)\n    )\n    raw = -model.fit(features[FEATURES]).decision_function(features[FEATURES])\n    # Percentile is easier to interpret consistently; it is NOT a probability.\n    score = pd.Series(raw).rank(pct=True).mul(100).round(1)\n    level = np.where(score >= 90, "High", np.where(score >= 75, "Elevated", "Typical"))\n    note = np.where(\n        score >= 90,\n        "Unusual profile pattern; verify documents and references before onboarding.",\n        np.where(score >= 75, "Some profile metrics differ from peers; consider additional checks.",\n                 "No strong anomaly signal relative to this synthetic supplier set.")\n    )\n    return pd.DataFrame({\n        "supplier_id": features["supplier_id"].values,\n        "ml_anomaly_score": score.values,\n        "ml_risk_level": level,\n        "ml_risk_note": note,\n    })\n'
Path("ml_risk_model.py").write_text(ml_risk_source, encoding="utf-8")
engine_source = 'import pandas as pd\nfrom ml_risk_model import score_supplier_anomalies\n\n\n# ============================================================\n# COMPLETE INDIA TRANSIT MATRIX\n# ============================================================\n\nfrom india_locations import INDIA_LOCATIONS\n\nREGION = {\'Chandigarh\': \'North\', \'Delhi\': \'North\', \'Haryana\': \'North\', \'Himachal Pradesh\': \'North\', \'Jammu and Kashmir\': \'North\', \'Ladakh\': \'North\', \'Punjab\': \'North\', \'Rajasthan\': \'North\', \'Uttarakhand\': \'North\', \'Uttar Pradesh\': \'North\', \'Goa\': \'West\', \'Gujarat\': \'West\', \'Maharashtra\': \'West\', \'Madhya Pradesh\': \'West\', \'Chhattisgarh\': \'Central\', \'Jharkhand\': \'Central\', \'Bihar\': \'Central\', \'Odisha\': \'East\', \'West Bengal\': \'East\', \'Sikkim\': \'East\', \'Andhra Pradesh\': \'South\', \'Karnataka\': \'South\', \'Kerala\': \'South\', \'Tamil Nadu\': \'South\', \'Telangana\': \'South\', \'Puducherry\': \'South\', \'Arunachal Pradesh\': \'Northeast\', \'Assam\': \'Northeast\', \'Manipur\': \'Northeast\', \'Meghalaya\': \'Northeast\', \'Mizoram\': \'Northeast\', \'Nagaland\': \'Northeast\', \'Tripura\': \'Northeast\', \'Andaman and Nicobar Islands\': \'Island\', \'Lakshadweep\': \'Island\', \'Dadra and Nagar Haveli and Daman and Diu\': \'West\'}\n\nREGIONAL_TRANSIT = {(\'North\', \'North\'): 2, (\'North\', \'West\'): 3, (\'North\', \'Central\'): 3, (\'North\', \'East\'): 4, (\'North\', \'South\'): 5, (\'North\', \'Northeast\'): 6, (\'North\', \'Island\'): 8, (\'West\', \'North\'): 3, (\'West\', \'West\'): 2, (\'West\', \'Central\'): 3, (\'West\', \'East\'): 4, (\'West\', \'South\'): 4, (\'West\', \'Northeast\'): 6, (\'West\', \'Island\'): 8, (\'Central\', \'North\'): 3, (\'Central\', \'West\'): 3, (\'Central\', \'Central\'): 2, (\'Central\', \'East\'): 3, (\'Central\', \'South\'): 4, (\'Central\', \'Northeast\'): 5, (\'Central\', \'Island\'): 8, (\'East\', \'North\'): 4, (\'East\', \'West\'): 4, (\'East\', \'Central\'): 3, (\'East\', \'East\'): 2, (\'East\', \'South\'): 4, (\'East\', \'Northeast\'): 4, (\'East\', \'Island\'): 8, (\'South\', \'North\'): 5, (\'South\', \'West\'): 4, (\'South\', \'Central\'): 4, (\'South\', \'East\'): 4, (\'South\', \'South\'): 2, (\'South\', \'Northeast\'): 6, (\'South\', \'Island\'): 8, (\'Northeast\', \'North\'): 6, (\'Northeast\', \'West\'): 6, (\'Northeast\', \'Central\'): 5, (\'Northeast\', \'East\'): 4, (\'Northeast\', \'South\'): 6, (\'Northeast\', \'Northeast\'): 2, (\'Northeast\', \'Island\'): 9, (\'Island\', \'North\'): 8, (\'Island\', \'West\'): 8, (\'Island\', \'Central\'): 8, (\'Island\', \'East\'): 8, (\'Island\', \'South\'): 8, (\'Island\', \'Northeast\'): 9, (\'Island\', \'Island\'): 6}\n\nTRANSIT_DAYS = {\'Andhra Pradesh\': {\'Andhra Pradesh\': 1, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 2, \'Kerala\': 2, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 2, \'Telangana\': 2, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 2}, \'Arunachal Pradesh\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 1, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 2, \'Mizoram\': 2, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Assam\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 1, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 2, \'Mizoram\': 2, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Bihar\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 5, \'Assam\': 5, \'Bihar\': 1, \'Chhattisgarh\': 2, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 2, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 5, \'Meghalaya\': 5, \'Mizoram\': 5, \'Nagaland\': 5, \'Odisha\': 3, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 3, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 5, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 3, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Chhattisgarh\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 5, \'Assam\': 5, \'Bihar\': 2, \'Chhattisgarh\': 1, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 2, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 5, \'Meghalaya\': 5, \'Mizoram\': 5, \'Nagaland\': 5, \'Odisha\': 3, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 3, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 5, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 3, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Goa\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 1, \'Gujarat\': 2, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 2, \'Maharashtra\': 2, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 4, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 6, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 2, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Gujarat\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 2, \'Gujarat\': 1, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 2, \'Maharashtra\': 2, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 4, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 6, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 2, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Haryana\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 1, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Himachal Pradesh\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 1, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Jharkhand\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 5, \'Assam\': 5, \'Bihar\': 2, \'Chhattisgarh\': 2, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 1, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 5, \'Meghalaya\': 5, \'Mizoram\': 5, \'Nagaland\': 5, \'Odisha\': 3, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 3, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 5, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 3, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Karnataka\': {\'Andhra Pradesh\': 2, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 1, \'Kerala\': 2, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 2, \'Telangana\': 2, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 2}, \'Kerala\': {\'Andhra Pradesh\': 2, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 2, \'Kerala\': 1, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 2, \'Telangana\': 2, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 2}, \'Madhya Pradesh\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 2, \'Gujarat\': 2, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 1, \'Maharashtra\': 2, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 4, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 6, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 2, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Maharashtra\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 2, \'Gujarat\': 2, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 2, \'Maharashtra\': 1, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 4, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 6, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 2, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Manipur\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 1, \'Meghalaya\': 2, \'Mizoram\': 2, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Meghalaya\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 1, \'Mizoram\': 2, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Mizoram\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 2, \'Mizoram\': 1, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Nagaland\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 2, \'Mizoram\': 2, \'Nagaland\': 1, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 2, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Odisha\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 4, \'Assam\': 4, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 4, \'Himachal Pradesh\': 4, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 4, \'Meghalaya\': 4, \'Mizoram\': 4, \'Nagaland\': 4, \'Odisha\': 1, \'Punjab\': 4, \'Rajasthan\': 4, \'Sikkim\': 2, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 4, \'Uttar Pradesh\': 4, \'Uttarakhand\': 4, \'West Bengal\': 2, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 4, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 4, \'Jammu and Kashmir\': 4, \'Ladakh\': 4, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Punjab\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 1, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Rajasthan\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 1, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Sikkim\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 4, \'Assam\': 4, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 4, \'Himachal Pradesh\': 4, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 4, \'Meghalaya\': 4, \'Mizoram\': 4, \'Nagaland\': 4, \'Odisha\': 2, \'Punjab\': 4, \'Rajasthan\': 4, \'Sikkim\': 1, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 4, \'Uttar Pradesh\': 4, \'Uttarakhand\': 4, \'West Bengal\': 2, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 4, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 4, \'Jammu and Kashmir\': 4, \'Ladakh\': 4, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Tamil Nadu\': {\'Andhra Pradesh\': 2, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 2, \'Kerala\': 2, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 1, \'Telangana\': 2, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 2}, \'Telangana\': {\'Andhra Pradesh\': 2, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 2, \'Kerala\': 2, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 2, \'Telangana\': 1, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 2}, \'Tripura\': {\'Andhra Pradesh\': 6, \'Arunachal Pradesh\': 2, \'Assam\': 2, \'Bihar\': 5, \'Chhattisgarh\': 5, \'Goa\': 6, \'Gujarat\': 6, \'Haryana\': 6, \'Himachal Pradesh\': 6, \'Jharkhand\': 5, \'Karnataka\': 6, \'Kerala\': 6, \'Madhya Pradesh\': 6, \'Maharashtra\': 6, \'Manipur\': 2, \'Meghalaya\': 2, \'Mizoram\': 2, \'Nagaland\': 2, \'Odisha\': 4, \'Punjab\': 6, \'Rajasthan\': 6, \'Sikkim\': 4, \'Tamil Nadu\': 6, \'Telangana\': 6, \'Tripura\': 1, \'Uttar Pradesh\': 6, \'Uttarakhand\': 6, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 9, \'Chandigarh\': 6, \'Dadra and Nagar Haveli and Daman and Diu\': 6, \'Delhi\': 6, \'Jammu and Kashmir\': 6, \'Ladakh\': 6, \'Lakshadweep\': 9, \'Puducherry\': 6}, \'Uttar Pradesh\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 1, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Uttarakhand\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 1, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'West Bengal\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 4, \'Assam\': 4, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 4, \'Himachal Pradesh\': 4, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 4, \'Meghalaya\': 4, \'Mizoram\': 4, \'Nagaland\': 4, \'Odisha\': 2, \'Punjab\': 4, \'Rajasthan\': 4, \'Sikkim\': 2, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 4, \'Uttar Pradesh\': 4, \'Uttarakhand\': 4, \'West Bengal\': 1, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 4, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 4, \'Jammu and Kashmir\': 4, \'Ladakh\': 4, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Andaman and Nicobar Islands\': {\'Andhra Pradesh\': 8, \'Arunachal Pradesh\': 9, \'Assam\': 9, \'Bihar\': 8, \'Chhattisgarh\': 8, \'Goa\': 8, \'Gujarat\': 8, \'Haryana\': 8, \'Himachal Pradesh\': 8, \'Jharkhand\': 8, \'Karnataka\': 8, \'Kerala\': 8, \'Madhya Pradesh\': 8, \'Maharashtra\': 8, \'Manipur\': 9, \'Meghalaya\': 9, \'Mizoram\': 9, \'Nagaland\': 9, \'Odisha\': 8, \'Punjab\': 8, \'Rajasthan\': 8, \'Sikkim\': 8, \'Tamil Nadu\': 8, \'Telangana\': 8, \'Tripura\': 9, \'Uttar Pradesh\': 8, \'Uttarakhand\': 8, \'West Bengal\': 8, \'Andaman and Nicobar Islands\': 1, \'Chandigarh\': 8, \'Dadra and Nagar Haveli and Daman and Diu\': 8, \'Delhi\': 8, \'Jammu and Kashmir\': 8, \'Ladakh\': 8, \'Lakshadweep\': 6, \'Puducherry\': 8}, \'Chandigarh\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 1, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Dadra and Nagar Haveli and Daman and Diu\': {\'Andhra Pradesh\': 4, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 2, \'Gujarat\': 2, \'Haryana\': 3, \'Himachal Pradesh\': 3, \'Jharkhand\': 3, \'Karnataka\': 4, \'Kerala\': 4, \'Madhya Pradesh\': 2, \'Maharashtra\': 2, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 3, \'Rajasthan\': 3, \'Sikkim\': 4, \'Tamil Nadu\': 4, \'Telangana\': 4, \'Tripura\': 6, \'Uttar Pradesh\': 3, \'Uttarakhand\': 3, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 3, \'Dadra and Nagar Haveli and Daman and Diu\': 1, \'Delhi\': 3, \'Jammu and Kashmir\': 3, \'Ladakh\': 3, \'Lakshadweep\': 8, \'Puducherry\': 4}, \'Delhi\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 1, \'Jammu and Kashmir\': 2, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Jammu and Kashmir\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 1, \'Ladakh\': 2, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Ladakh\': {\'Andhra Pradesh\': 5, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 3, \'Chhattisgarh\': 3, \'Goa\': 3, \'Gujarat\': 3, \'Haryana\': 2, \'Himachal Pradesh\': 2, \'Jharkhand\': 3, \'Karnataka\': 5, \'Kerala\': 5, \'Madhya Pradesh\': 3, \'Maharashtra\': 3, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 2, \'Rajasthan\': 2, \'Sikkim\': 4, \'Tamil Nadu\': 5, \'Telangana\': 5, \'Tripura\': 6, \'Uttar Pradesh\': 2, \'Uttarakhand\': 2, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 2, \'Dadra and Nagar Haveli and Daman and Diu\': 3, \'Delhi\': 2, \'Jammu and Kashmir\': 2, \'Ladakh\': 1, \'Lakshadweep\': 8, \'Puducherry\': 5}, \'Lakshadweep\': {\'Andhra Pradesh\': 8, \'Arunachal Pradesh\': 9, \'Assam\': 9, \'Bihar\': 8, \'Chhattisgarh\': 8, \'Goa\': 8, \'Gujarat\': 8, \'Haryana\': 8, \'Himachal Pradesh\': 8, \'Jharkhand\': 8, \'Karnataka\': 8, \'Kerala\': 8, \'Madhya Pradesh\': 8, \'Maharashtra\': 8, \'Manipur\': 9, \'Meghalaya\': 9, \'Mizoram\': 9, \'Nagaland\': 9, \'Odisha\': 8, \'Punjab\': 8, \'Rajasthan\': 8, \'Sikkim\': 8, \'Tamil Nadu\': 8, \'Telangana\': 8, \'Tripura\': 9, \'Uttar Pradesh\': 8, \'Uttarakhand\': 8, \'West Bengal\': 8, \'Andaman and Nicobar Islands\': 6, \'Chandigarh\': 8, \'Dadra and Nagar Haveli and Daman and Diu\': 8, \'Delhi\': 8, \'Jammu and Kashmir\': 8, \'Ladakh\': 8, \'Lakshadweep\': 1, \'Puducherry\': 8}, \'Puducherry\': {\'Andhra Pradesh\': 2, \'Arunachal Pradesh\': 6, \'Assam\': 6, \'Bihar\': 4, \'Chhattisgarh\': 4, \'Goa\': 4, \'Gujarat\': 4, \'Haryana\': 5, \'Himachal Pradesh\': 5, \'Jharkhand\': 4, \'Karnataka\': 2, \'Kerala\': 2, \'Madhya Pradesh\': 4, \'Maharashtra\': 4, \'Manipur\': 6, \'Meghalaya\': 6, \'Mizoram\': 6, \'Nagaland\': 6, \'Odisha\': 4, \'Punjab\': 5, \'Rajasthan\': 5, \'Sikkim\': 4, \'Tamil Nadu\': 2, \'Telangana\': 2, \'Tripura\': 6, \'Uttar Pradesh\': 5, \'Uttarakhand\': 5, \'West Bengal\': 4, \'Andaman and Nicobar Islands\': 8, \'Chandigarh\': 5, \'Dadra and Nagar Haveli and Daman and Diu\': 4, \'Delhi\': 5, \'Jammu and Kashmir\': 5, \'Ladakh\': 5, \'Lakshadweep\': 8, \'Puducherry\': 1}}\n\n\ndef prepare_delivery_df(listing_df, supplier_df, destination):\n\n    delivery_df = listing_df.merge(\n        supplier_df[["supplier_id", "state"]],\n        on="supplier_id",\n        how="left"\n    )\n\n    delivery_df["transit_days"] = delivery_df["state"].apply(\n        lambda supplier_state: TRANSIT_DAYS[supplier_state][destination]\n    )\n\n    delivery_df["effective_delivery_days"] = (\n        delivery_df["dispatch_days"]\n        + delivery_df["transit_days"]\n    )\n\n    return delivery_df\n\n\n# ============================================================\n# HARD CONSTRAINTS\n# ============================================================\n\ndef apply_hard_constraints(delivery_df, requirement):\n\n    filtered = delivery_df.copy()\n\n    filtered = filtered[\n        filtered["variant"].str.contains(\n            requirement["variant"],\n            case=False,\n            na=False\n        )\n    ]\n\n    filtered = filtered[\n        filtered["price_inr"] <= requirement["max_price"]\n    ]\n\n    filtered = filtered[\n        filtered["price_unit"] == requirement["price_unit"]\n    ]\n\n    filtered = filtered[\n        filtered["available_quantity"] >= requirement["quantity"]\n    ]\n\n    filtered = filtered[\n        filtered["effective_delivery_days"]\n        <= requirement["max_delivery_days"]\n    ]\n\n    filtered = filtered[\n        filtered["rating"] >= requirement["min_rating"]\n    ]\n\n    filtered = filtered[\n        filtered["warranty_months"]\n        >= requirement["min_warranty_years"] * 12\n    ]\n\n    return filtered.reset_index(drop=True)\n\n\n# ============================================================\n# WEIGHTS\n# ============================================================\n\ndef create_weights(\n    price_priority,\n    delivery_priority,\n    quality_priority,\n    reliability_priority\n):\n\n    total = (\n        price_priority\n        + delivery_priority\n        + quality_priority\n        + reliability_priority\n    )\n\n    if total == 0:\n        raise ValueError(\n            "At least one priority must be greater than 0."\n        )\n\n    return {\n        "price": price_priority / total,\n        "delivery": delivery_priority / total,\n        "quality": quality_priority / total,\n        "reliability": reliability_priority / total\n    }\n\n\n# ============================================================\n# SCORING\n# ============================================================\n\ndef calculate_supplier_scores(df, weights):\n\n    scored = df.copy()\n\n    price_range = (\n        scored["price_inr"].max()\n        - scored["price_inr"].min()\n    )\n\n    delivery_range = (\n        scored["effective_delivery_days"].max()\n        - scored["effective_delivery_days"].min()\n    )\n\n    quality_range = (\n        scored["rating"].max()\n        - scored["rating"].min()\n    )\n\n    scored["price_score"] = (\n        (scored["price_inr"].max() - scored["price_inr"])\n        / (price_range if price_range != 0 else 1)\n    )\n\n    scored["delivery_score"] = (\n        (\n            scored["effective_delivery_days"].max()\n            - scored["effective_delivery_days"]\n        )\n        / (delivery_range if delivery_range != 0 else 1)\n    )\n\n    scored["quality_score"] = (\n        (scored["rating"] - scored["rating"].min())\n        / (quality_range if quality_range != 0 else 1)\n    )\n\n    scored["reliability_score"] = (\n        scored["response_rate_pct"]\n        + scored["on_time_delivery_pct"]\n    ) / 200\n\n    scored["overall_score"] = (\n        scored["price_score"] * weights["price"]\n        + scored["delivery_score"] * weights["delivery"]\n        + scored["quality_score"] * weights["quality"]\n        + scored["reliability_score"] * weights["reliability"]\n    )\n\n    scored = scored.sort_values(\n        "overall_score",\n        ascending=False\n    ).reset_index(drop=True)\n\n    scored["rank"] = scored.index + 1\n\n    return scored\n\n\ndef rank_suppliers(\n    df,\n    price_priority,\n    delivery_priority,\n    quality_priority,\n    reliability_priority\n):\n\n    weights = create_weights(\n        price_priority,\n        delivery_priority,\n        quality_priority,\n        reliability_priority\n    )\n\n    ranked_df = calculate_supplier_scores(\n        df,\n        weights\n    )\n\n    return ranked_df, weights\n\n\n# ============================================================\n# FAILURE REASONS\n# ============================================================\n\ndef get_failure_reasons(row, requirement):\n\n    reasons = []\n\n    if row["price_gap"] > 0:\n        reasons.append(\n            f"Price is ₹{row[\'price_gap\']:.2f} above the maximum"\n        )\n\n    if row["delivery_gap"] > 0:\n        reasons.append(\n            f"Delivery is {int(row[\'delivery_gap\'])} day(s) beyond the limit"\n        )\n\n    if row["rating_gap"] > 0:\n        reasons.append(\n            f"Rating is {row[\'rating_gap\']:.2f} points below the minimum"\n        )\n\n    if row["warranty_gap_months"] > 0:\n        reasons.append(\n            f"Warranty is {int(row[\'warranty_gap_months\'])} month(s) below the minimum"\n        )\n\n    if row["stock_gap"] > 0:\n        reasons.append(\n            f"Stock is {int(row[\'stock_gap\'])} units below the required quantity"\n        )\n\n    if row["unit_fail"]:\n        reasons.append(\n            f"Price unit does not match the requested unit "\n            f"({requirement[\'price_unit\']})"\n        )\n\n    return reasons\n\n\n# ============================================================\n# NEAR MATCHES\n# ============================================================\n\ndef find_near_matches(delivery_df, requirement, top_n=5):\n\n    variant_df = delivery_df[\n        delivery_df["variant"].str.contains(\n            requirement["variant"],\n            case=False,\n            na=False\n        )\n    ].copy()\n\n    if variant_df.empty:\n        return pd.DataFrame()\n\n    variant_df["price_fail"] = (\n        variant_df["price_inr"] > requirement["max_price"]\n    )\n\n    variant_df["unit_fail"] = (\n        variant_df["price_unit"] != requirement["price_unit"]\n    )\n\n    variant_df["stock_fail"] = (\n        variant_df["available_quantity"] < requirement["quantity"]\n    )\n\n    variant_df["delivery_fail"] = (\n        variant_df["effective_delivery_days"]\n        > requirement["max_delivery_days"]\n    )\n\n    variant_df["rating_fail"] = (\n        variant_df["rating"] < requirement["min_rating"]\n    )\n\n    variant_df["warranty_fail"] = (\n        variant_df["warranty_months"]\n        < requirement["min_warranty_years"] * 12\n    )\n\n    variant_df["total_failures"] = (\n        variant_df["price_fail"].astype(int)\n        + variant_df["unit_fail"].astype(int)\n        + variant_df["stock_fail"].astype(int)\n        + variant_df["delivery_fail"].astype(int)\n        + variant_df["rating_fail"].astype(int)\n        + variant_df["warranty_fail"].astype(int)\n    )\n\n    variant_df["price_gap"] = (\n        variant_df["price_inr"]\n        - requirement["max_price"]\n    ).clip(lower=0)\n\n    variant_df["delivery_gap"] = (\n        variant_df["effective_delivery_days"]\n        - requirement["max_delivery_days"]\n    ).clip(lower=0)\n\n    variant_df["rating_gap"] = (\n        requirement["min_rating"]\n        - variant_df["rating"]\n    ).clip(lower=0)\n\n    variant_df["warranty_gap_months"] = (\n        requirement["min_warranty_years"] * 12\n        - variant_df["warranty_months"]\n    ).clip(lower=0)\n\n    variant_df["stock_gap"] = (\n        requirement["quantity"]\n        - variant_df["available_quantity"]\n    ).clip(lower=0)\n\n    variant_df["price_gap_score"] = (\n        variant_df["price_gap"]\n        / requirement["max_price"]\n    )\n\n    variant_df["delivery_gap_score"] = (\n        variant_df["delivery_gap"]\n        / requirement["max_delivery_days"]\n    )\n\n    variant_df["rating_gap_score"] = (\n        variant_df["rating_gap"]\n        / requirement["min_rating"]\n    )\n\n    variant_df["warranty_gap_score"] = (\n        variant_df["warranty_gap_months"]\n        / (requirement["min_warranty_years"] * 12)\n    )\n\n    variant_df["stock_gap_score"] = (\n        variant_df["stock_gap"]\n        / requirement["quantity"]\n    )\n\n    variant_df["total_gap_score"] = (\n        variant_df["price_gap_score"]\n        + variant_df["delivery_gap_score"]\n        + variant_df["rating_gap_score"]\n        + variant_df["warranty_gap_score"]\n        + variant_df["stock_gap_score"]\n    )\n\n    variant_df["failure_reasons"] = variant_df.apply(\n        lambda row: get_failure_reasons(\n            row,\n            requirement\n        ),\n        axis=1\n    )\n\n    near_matches = variant_df[\n        variant_df["total_failures"] > 0\n    ].copy()\n\n    near_matches = near_matches.sort_values(\n        ["total_failures", "total_gap_score"],\n        ascending=[True, True]\n    ).reset_index(drop=True)\n\n    near_matches["near_match_rank"] = (\n        near_matches.index + 1\n    )\n\n    return near_matches.head(top_n)\n\n\n# ============================================================\n# RISK FLAGS\n# ============================================================\n\ndef generate_risk_flags(row, requirement):\n\n    risks = []\n\n    if row["response_rate_pct"] < 80:\n        risks.append(\n            f"Low response rate ({row[\'response_rate_pct\']:.1f}%)"\n        )\n\n    if row["on_time_delivery_pct"] < 80:\n        risks.append(\n            f"Low on-time delivery rate "\n            f"({row[\'on_time_delivery_pct\']:.1f}%)"\n        )\n\n    stock_ratio = (\n        row["available_quantity"]\n        / requirement["quantity"]\n    )\n\n    if stock_ratio < 1.5:\n        risks.append(\n            f"Limited stock buffer "\n            f"({int(row[\'available_quantity\'])} available "\n            f"for {int(requirement[\'quantity\'])} required)"\n        )\n\n    if not row["verified_supplier"]:\n        risks.append(\n            "Supplier is not verified"\n        )\n\n    if row["warranty_months"] < 24:\n        risks.append(\n            f"Warranty is only "\n            f"{int(row[\'warranty_months\'])} months"\n        )\n\n    return risks\n\n\n# ============================================================\n# EXPLAINABILITY\n# ============================================================\n\ndef generate_explanation_v2(\n    ranked_df,\n    weights,\n    requirement\n):\n\n    recommended = ranked_df.iloc[0]\n\n    explanation = []\n\n    explanation.append("🏆 RECOMMENDATION")\n\n    explanation.append(\n        f"{recommended[\'supplier_name\']} "\n        f"is the recommended supplier."\n    )\n\n    priority_map = {\n        "price": weights["price"],\n        "delivery": weights["delivery"],\n        "quality": weights["quality"],\n        "reliability": weights["reliability"]\n    }\n\n    highest_priority = max(\n        priority_map,\n        key=priority_map.get\n    )\n\n    highest_priority_pct = (\n        priority_map[highest_priority] * 100\n    )\n\n    priority_names = {\n        "price": "price",\n        "delivery": "delivery speed",\n        "quality": "supplier quality",\n        "reliability": "supplier reliability"\n    }\n\n    explanation.append(\n        f"Your highest priority is "\n        f"{priority_names[highest_priority]} "\n        f"({highest_priority_pct:.0f}% weight)."\n    )\n\n    score_column = f"{highest_priority}_score"\n\n    explanation.append(\n        f"The supplier scored "\n        f"{recommended[score_column]:.2f} "\n        f"on this priority."\n    )\n\n    explanation.append("")\n    explanation.append("✅ REQUIREMENT FIT")\n\n    explanation.append(\n        f"Product: {recommended[\'variant\']}"\n    )\n\n    explanation.append(\n        f"Price: ₹{recommended[\'price_inr\']:.2f} "\n        f"{recommended[\'price_unit\']} "\n        f"(maximum ₹{requirement[\'max_price\']:.2f})"\n    )\n\n    explanation.append(\n        f"Available stock: "\n        f"{int(recommended[\'available_quantity\'])} "\n        f"(required {int(requirement[\'quantity\'])})"\n    )\n\n    explanation.append(\n        f"Effective delivery: "\n        f"{int(recommended[\'effective_delivery_days\'])} days "\n        f"(maximum {int(requirement[\'max_delivery_days\'])} days)"\n    )\n\n    explanation.append(\n        f"Rating: {recommended[\'rating\']:.2f}/5 "\n        f"(minimum {requirement[\'min_rating\']:.1f})"\n    )\n\n    explanation.append(\n        f"Warranty: "\n        f"{int(recommended[\'warranty_months\'])} months "\n        f"(minimum "\n        f"{int(requirement[\'min_warranty_years\'] * 12)} months)"\n    )\n\n    explanation.append("")\n    explanation.append("💪 STRENGTHS")\n\n    if recommended["price_score"] == ranked_df["price_score"].max():\n        explanation.append(\n            "• Strongest price position among eligible suppliers."\n        )\n\n    if recommended["delivery_score"] == ranked_df["delivery_score"].max():\n        explanation.append(\n            "• Fastest effective delivery among eligible suppliers."\n        )\n\n    if recommended["quality_score"] == ranked_df["quality_score"].max():\n        explanation.append(\n            "• Highest supplier rating among eligible suppliers."\n        )\n\n    if recommended["reliability_score"] == ranked_df["reliability_score"].max():\n        explanation.append(\n            "• Strongest reliability score among eligible suppliers."\n        )\n\n    explanation.append("")\n    explanation.append("⚖️ TRADE-OFFS")\n\n    if len(ranked_df) > 1:\n\n        alternative = ranked_df.iloc[1]\n\n        price_diff = (\n            recommended["price_inr"]\n            - alternative["price_inr"]\n        )\n\n        delivery_diff = (\n            recommended["effective_delivery_days"]\n            - alternative["effective_delivery_days"]\n        )\n\n        rating_diff = (\n            recommended["rating"]\n            - alternative["rating"]\n        )\n\n        warranty_diff = (\n            recommended["warranty_months"]\n            - alternative["warranty_months"]\n        )\n\n        if price_diff > 0:\n            explanation.append(\n                f"• {alternative[\'supplier_name\']} "\n                f"is ₹{abs(price_diff):.2f} cheaper."\n            )\n\n        elif price_diff < 0:\n            explanation.append(\n                f"• {recommended[\'supplier_name\']} "\n                f"is ₹{abs(price_diff):.2f} cheaper."\n            )\n\n        if delivery_diff > 0:\n            explanation.append(\n                f"• {alternative[\'supplier_name\']} "\n                f"is {abs(delivery_diff):.0f} day(s) faster."\n            )\n\n        elif delivery_diff < 0:\n            explanation.append(\n                f"• {recommended[\'supplier_name\']} "\n                f"is {abs(delivery_diff):.0f} day(s) faster."\n            )\n\n        if rating_diff > 0:\n            explanation.append(\n                f"• Recommended supplier has a "\n                f"{abs(rating_diff):.2f} point higher rating."\n            )\n\n        elif rating_diff < 0:\n            explanation.append(\n                f"• {alternative[\'supplier_name\']} "\n                f"has a {abs(rating_diff):.2f} point higher rating."\n            )\n\n        if warranty_diff > 0:\n            explanation.append(\n                f"• Recommended supplier offers "\n                f"{abs(warranty_diff):.0f} more months of warranty."\n            )\n\n        elif warranty_diff < 0:\n            explanation.append(\n                f"• {alternative[\'supplier_name\']} "\n                f"offers {abs(warranty_diff):.0f} more months of warranty."\n            )\n\n    return "\\\n".join(explanation)\n\n\n# ============================================================\n# MACHINE-LEARNING RISK ENRICHMENT\n# ============================================================\ndef attach_ml_risk(df, supplier_df, listing_df):\n    """Add unsupervised anomaly flags; this does not alter hard-constraint decisions."""\n    if df.empty:\n        return df\n    ml_scores = score_supplier_anomalies(supplier_df, listing_df)\n    enriched = df.merge(ml_scores, on="supplier_id", how="left")\n    return enriched\n\n\n# ============================================================\n# MAIN COPILOT\n# ============================================================\n\ndef run_supplier_copilot_dynamic(\n    listing_df,\n    supplier_df,\n    requirement,\n    price_priority=25,\n    delivery_priority=25,\n    quality_priority=25,\n    reliability_priority=25,\n    top_n=5\n):\n\n    delivery_df = prepare_delivery_df(\n        listing_df,\n        supplier_df,\n        requirement["location"]\n    )\n\n    eligible_df = apply_hard_constraints(\n        delivery_df,\n        requirement\n    )\n\n    if eligible_df.empty:\n\n        near_matches = find_near_matches(\n            delivery_df,\n            requirement,\n            top_n=top_n\n        )\n\n        near_matches = attach_ml_risk(near_matches, supplier_df, listing_df)\n        explanation = (\n            "No supplier satisfies every hard constraint. The closest alternatives "\n            "below are ordered by the number and severity of requirement gaps. "\n            "ML risk is a separate anomaly-screening signal based on supplier-profile "\n            "patterns; it is not a probability of fraud or failure."\n        )\n        return {\n            "status": "NO_PERFECT_MATCH",\n            "ranked_suppliers": pd.DataFrame(),\n            "weights": None,\n            "explanation": explanation,\n            "near_matches": near_matches\n        }\n\n    ranked_df, weights = rank_suppliers(\n        eligible_df,\n        price_priority,\n        delivery_priority,\n        quality_priority,\n        reliability_priority\n    )\n\n    explanation = generate_explanation_v2(\n        ranked_df,\n        weights,\n        requirement\n    )\n\n    ranked_df = ranked_df.copy()\n\n    ranked_df["risk_flags"] = ranked_df.apply(\n        lambda row: generate_risk_flags(\n            row,\n            requirement\n        ),\n        axis=1\n    )\n    ranked_df = attach_ml_risk(ranked_df, supplier_df, listing_df)\n\n    return {\n        "status": "MATCH_FOUND",\n        "ranked_suppliers": ranked_df,\n        "weights": weights,\n        "explanation": explanation,\n        "near_matches": pd.DataFrame()\n    }'
Path("copilot_engine.py").write_text(engine_source, encoding="utf-8")
exec(compile(Path("copilot_engine.py").read_text(encoding="utf-8"), "copilot_engine.py", "exec"))
print("Wrote ml_risk_model.py and copilot_engine.py")


India location master created successfully.
Total States + UTs: 36

States + UTs:
01. Andhra Pradesh
02. Arunachal Pradesh
03. Assam
04. Bihar
05. Chhattisgarh
06. Goa
07. Gujarat
08. Haryana
09. Himachal Pradesh
10. Jharkhand
11. Karnataka
12. Kerala
13. Madhya Pradesh
14. Maharashtra
15. Manipur
16. Meghalaya
17. Mizoram
18. Nagaland
19. Odisha
20. Punjab
21. Rajasthan
22. Sikkim
23. Tamil Nadu
24. Telangana
25. Tripura
26. Uttar Pradesh
27. Uttarakhand
28. West Bengal
29. Andaman and Nicobar Islands
30. Chandigarh
31. Dadra and Nagar Haveli and Daman and Diu
32. Delhi
33. Jammu and Kashmir
34. Ladakh
35. Lakshadweep
36. Puducherry
Wrote ml_risk_model.py and copilot_engine.py


## 4. Write the latest Streamlit application

## LLM natural-language procurement assistant
The app now accepts a free-text procurement request and uses an OpenAI LLM to extract structured fields. Configure `OPENAI_API_KEY` in the app sidebar or as a secret. Supplier ranking remains deterministic.


In [15]:
from pathlib import Path
app_source = 'import streamlit as st\nimport pandas as pd\n\nfrom copilot_engine import run_supplier_copilot_dynamic\nfrom india_locations import INDIA_LOCATIONS\nfrom llm_procurement_assistant import parse_procurement_request\nimport os\n\n\n# ============================================================\n# PAGE CONFIG\n# ============================================================\n\nst.set_page_config(\n    page_title="AI B2B Supplier Selection Copilot",\n    page_icon="🤖",\n    layout="wide"\n)\n\n\n# ============================================================\n# LOAD DATA\n# ============================================================\n\nproduct_df = pd.read_csv("product_catalogue.csv")\nsupplier_df = pd.read_csv("supplier_master.csv")\nlisting_df = pd.read_csv("supplier_listings.csv")\n\n\n# ============================================================\n# HELPER: PRODUCT OPTIONS\n# ============================================================\n\nproduct_options = (\n    listing_df[\n        ["category", "product_family", "variant"]\n    ]\n    .drop_duplicates()\n    .sort_values(\n        ["category", "product_family", "variant"]\n    )\n    .reset_index(drop=True)\n)\n\n\n# ============================================================\n# HEADER\n# ============================================================\n\nst.title("🤖 AI B2B Supplier Selection Copilot")\n\nst.caption(\n    "Turn procurement requirements into explainable supplier recommendations."\n)\n\nst.divider()\n\n\n# ============================================================\n# DATASET STATUS\n# ============================================================\n\ncol1, col2, col3 = st.columns(3)\n\nwith col1:\n    st.metric(\n        "Suppliers",\n        f"{supplier_df[\'supplier_id\'].nunique():,}"\n    )\n\nwith col2:\n    st.metric(\n        "Supplier Listings",\n        f"{len(listing_df):,}"\n    )\n\nwith col3:\n    st.metric(\n        "Product Variants",\n        f"{listing_df[\'variant\'].nunique():,}"\n    )\n\n\nst.divider()\n\n\n# ============================================================\n# LLM NATURAL-LANGUAGE PROCUREMENT INTAKE\n# ============================================================\n\nwith st.sidebar:\n    st.header("LLM Assistant Settings")\n    api_key_input = st.text_input(\n        "OpenAI API key",\n        type="password",\n        value="",\n        help="Optional. Used only for this session. For deployment, configure OPENAI_API_KEY as a secret.",\n        key="openai_api_key_input",\n        placeholder="sk-...",\n    )\n    st.caption("The LLM extracts requirements. Supplier filtering and ranking remain deterministic and explainable.")\n\nif "llm_parsed_request" not in st.session_state:\n    st.session_state["llm_parsed_request"] = None\n\nwith st.expander("💬 Describe your procurement need in plain English (LLM assistant)", expanded=True):\n    st.write("Example: *I need 500 units delivered to Maharashtra within 7 days, under ₹350 each, at least 4.5 stars and a 2-year warranty. Delivery is my top priority.*")\n    natural_request = st.text_area(\n        "Procurement request",\n        key="natural_language_request",\n        height=110,\n        placeholder="Describe product, quantity, budget, destination, delivery, quality and warranty...",\n    )\n    parse_clicked = st.button("✨ Interpret request with AI", type="secondary", key="parse_llm_request")\n    if parse_clicked:\n        try:\n            with st.spinner("Interpreting your request with the LLM..."):\n                parsed_request = parse_procurement_request(\n                    natural_request,\n                    valid_variants=product_options["variant"].tolist(),\n                    valid_price_units=sorted(listing_df["price_unit"].dropna().unique().tolist()),\n                    valid_locations=INDIA_LOCATIONS,\n                    api_key=api_key_input.strip() or os.getenv("OPENAI_API_KEY"),\n                )\n            st.session_state["llm_parsed_request"] = parsed_request\n            st.session_state["llm_parse_error"] = None\n        except Exception as exc:\n            st.session_state["llm_parse_error"] = str(exc)\n    if st.session_state.get("llm_parse_error"):\n        st.error(st.session_state["llm_parse_error"])\n    parsed_request = st.session_state.get("llm_parsed_request")\n    if parsed_request:\n        if parsed_request.get("interpreted_summary"):\n            st.info("**AI interpretation:** " + parsed_request["interpreted_summary"])\n        extracted = {k: v for k, v in parsed_request.items() if k in {\n            "variant", "quantity", "location", "max_price", "price_unit",\n            "max_delivery_days", "min_rating", "min_warranty_years",\n            "price_priority", "delivery_priority", "quality_priority", "reliability_priority"\n        } and v is not None}\n        if extracted:\n            st.write("**Extracted requirements**")\n            st.json(extracted)\n        questions = parsed_request.get("clarifying_questions", [])\n        if questions:\n            st.warning("Please confirm these details before relying on the result:")\n            for question in questions:\n                st.write(f"• {question}")\n        if st.button("Apply extracted values to the form", type="primary", key="apply_llm_values"):\n            widget_keys = {\n                "variant": "req_variant", "quantity": "req_quantity", "location": "req_destination",\n                "max_price": "req_max_price", "price_unit": "req_price_unit",\n                "max_delivery_days": "req_max_delivery_days", "min_rating": "req_min_rating",\n                "min_warranty_years": "req_min_warranty_years", "price_priority": "req_price_priority",\n                "delivery_priority": "req_delivery_priority", "quality_priority": "req_quality_priority",\n                "reliability_priority": "req_reliability_priority",\n            }\n            for field, widget_key in widget_keys.items():\n                value = parsed_request.get(field)\n                if value is not None:\n                    st.session_state[widget_key] = value\n            st.session_state["run_copilot"] = False\n            st.rerun()\n\n\n# ============================================================\n# REQUIREMENT INPUT\n# ============================================================\n\nst.header("1. Define Your Procurement Requirement")\n\ncol1, col2 = st.columns(2)\n\nwith col1:\n\n    variant = st.selectbox(\n        "Product / Variant",\n        options=product_options["variant"].tolist(),\n        key="req_variant"\n    )\n\n    quantity = st.number_input(\n        "Required Quantity",\n        min_value=1,\n        value=500,\n        step=1,\n        key="req_quantity"\n    )\n\n    destination_options = INDIA_LOCATIONS.copy()\n\n    destination = st.selectbox(\n        "Buyer Destination",\n        options=destination_options,\n        index=(\n            destination_options.index("Maharashtra")\n            if "Maharashtra" in destination_options\n            else 0\n        ),\n        key="req_destination"\n    )\n\nwith col2:\n\n    max_price = st.number_input(\n        "Maximum Price",\n        min_value=0.0,\n        value=350.0,\n        step=10.0,\n        key="req_max_price"\n    )\n\n    price_unit_options = sorted(\n        listing_df["price_unit"].dropna().unique().tolist()\n    )\n\n    default_unit = (\n        price_unit_options.index("per piece")\n        if "per piece" in price_unit_options\n        else 0\n    )\n\n    price_unit = st.selectbox(\n        "Price Unit",\n        options=price_unit_options,\n        index=default_unit,\n        key="req_price_unit"\n    )\n\n    max_delivery_days = st.number_input(\n        "Maximum Delivery",\n        min_value=1,\n        value=7,\n        step=1,\n        key="req_max_delivery_days"\n    )\n\ncol1, col2 = st.columns(2)\n\nwith col1:\n\n    min_rating = st.slider(\n        "Minimum Supplier Rating",\n        min_value=0.0,\n        max_value=5.0,\n        value=4.5,\n        step=0.1,\n        key="req_min_rating"\n    )\n\nwith col2:\n\n    min_warranty_years = st.number_input(\n        "Minimum Warranty (Years)",\n        min_value=0.0,\n        value=2.0,\n        step=0.5,\n        key="req_min_warranty_years"\n    )\n\n\n# ============================================================\n# BUYER PRIORITIES\n# ============================================================\n\nst.header("2. Set Your Priorities")\n\nst.caption(\n    "The Copilot uses these priorities to rank suppliers after hard constraints are applied."\n)\n\np1, p2, p3, p4 = st.columns(4)\n\nwith p1:\n    price_priority = st.slider(\n        "💰 Price",\n        min_value=0,\n        max_value=100,\n        value=20,\n        step=5,\n        key="req_price_priority"\n    )\n\nwith p2:\n    delivery_priority = st.slider(\n        "🚚 Delivery",\n        min_value=0,\n        max_value=100,\n        value=50,\n        step=5,\n        key="req_delivery_priority"\n    )\n\nwith p3:\n    quality_priority = st.slider(\n        "⭐ Quality",\n        min_value=0,\n        max_value=100,\n        value=20,\n        step=5,\n        key="req_quality_priority"\n    )\n\nwith p4:\n    reliability_priority = st.slider(\n        "🛡️ Reliability",\n        min_value=0,\n        max_value=100,\n        value=10,\n        step=5,\n        key="req_reliability_priority"\n    )\n\n\ntotal_priority = (\n    price_priority\n    + delivery_priority\n    + quality_priority\n    + reliability_priority\n)\n\nif total_priority == 0:\n    st.warning(\n        "Please give at least one priority a value greater than 0."\n    )\n\n\n# ============================================================\n# RUN COPILOT\n# ============================================================\n\nrun_copilot = st.button(\n    "🚀 Run Supplier Copilot",\n    type="primary",\n    width="stretch"\n)\n\n\n# ============================================================\n# RESULTS\n# ============================================================\n\nif run_copilot:\n    st.session_state["run_copilot"] = True\n\nif st.session_state.get("run_copilot", False):\n\n    if total_priority == 0:\n\n        st.error(\n            "At least one priority must be greater than 0."\n        )\n\n    else:\n\n        requirement = {\n            "raw_query": "",\n            "variant": variant,\n            "quantity": int(quantity),\n            "location": destination,\n            "max_price": float(max_price),\n            "price_unit": price_unit,\n            "max_delivery_days": int(max_delivery_days),\n            "min_rating": float(min_rating),\n            "min_warranty_years": float(min_warranty_years)\n        }\n\n        result = run_supplier_copilot_dynamic(\n            listing_df=listing_df,\n            supplier_df=supplier_df,\n            requirement=requirement,\n            price_priority=price_priority,\n            delivery_priority=delivery_priority,\n            quality_priority=quality_priority,\n            reliability_priority=reliability_priority,\n            top_n=5\n        )\n\n\n        # ====================================================\n        # MATCH FOUND\n        # ====================================================\n\n        if result["status"] == "MATCH_FOUND":\n\n            ranked_df = result["ranked_suppliers"]\n            recommended = ranked_df.iloc[0]\n\n            st.divider()\n            st.header("3. Supplier Recommendation")\n\n            st.success(\n                f"🏆 Recommended Supplier: **{recommended[\'supplier_name\']}**"\n            )\n\n            st.caption(\n                "Recommended from suppliers that satisfy all hard constraints, "\n                "then ranked according to your selected priorities."\n            )\n\n            c1, c2, c3, c4 = st.columns(4)\n\n            with c1:\n                st.metric(\n                    "Overall Score",\n                    f"{recommended[\'overall_score\']:.4f}"\n                )\n\n            with c2:\n                st.metric(\n                    "Price",\n                    f"₹{recommended[\'price_inr\']:,.2f}",\n                    help=(\n                        f"Quoted price {recommended[\'price_unit\']}"\n                    )\n                )\n\n            with c3:\n                st.metric(\n                    "Effective Delivery",\n                    f"{int(recommended[\'effective_delivery_days\'])} days",\n                    help=(\n                        f"Dispatch: {int(recommended[\'dispatch_days\'])} days + "\n                        f"Transit: {int(recommended[\'transit_days\'])} days"\n                    )\n                )\n\n            with c4:\n                st.metric(\n                    "Rating",\n                    f"{recommended[\'rating\']:.2f}/5"\n                )\n\n\n            # -----------------------------------------------\n            # SUPPLIER DETAILS\n            # -----------------------------------------------\n\n            st.subheader("Supplier Details")\n\n            d1, d2, d3, d4 = st.columns(4)\n\n            with d1:\n                st.caption("SUPPLIER")\n                st.write(f"**{recommended[\'supplier_name\']}**")\n\n            with d2:\n                st.caption("SUPPLIER ORIGIN")\n                st.write(f"📍 {recommended[\'state\']}")\n\n            with d3:\n                st.caption("AVAILABLE STOCK")\n                st.write(\n                    f"**{int(recommended[\'available_quantity\']):,} units**"\n                )\n\n            with d4:\n                st.caption("WARRANTY")\n                st.write(\n                    f"**{int(recommended[\'warranty_months\'])} months**"\n                )\n\n            st.caption(\n                "Supplier origin is separate from the buyer\'s destination. "\n                "Delivery time includes dispatch and estimated transit."\n            )\n\n\n            # -----------------------------------------------\n            # REQUIREMENT FIT\n            # -----------------------------------------------\n\n            st.subheader("Requirement Fit")\n\n            fit_data = pd.DataFrame({\n                "Requirement": [\n                    "Product",\n                    "Price",\n                    "Available Stock",\n                    "Effective Delivery",\n                    "Rating",\n                    "Warranty"\n                ],\n                "Required": [\n                    variant,\n                    f"≤ ₹{max_price:,.2f} {price_unit}",\n                    f"≥ {int(quantity):,}",\n                    f"≤ {int(max_delivery_days)} days",\n                    f"≥ {min_rating:.1f}",\n                    f"≥ {min_warranty_years:.1f} years"\n                ],\n                "Supplier": [\n                    recommended["variant"],\n                    f"₹{recommended[\'price_inr\']:,.2f} {recommended[\'price_unit\']}",\n                    f"{int(recommended[\'available_quantity\']):,}",\n                    f"{int(recommended[\'effective_delivery_days\'])} days",\n                    f"{recommended[\'rating\']:.2f}",\n                    f"{int(recommended[\'warranty_months\'])} months"\n                ],\n                "Status": [\n                    "PASS" if recommended["variant"] == variant else "CHECK",\n                    "PASS" if (\n                        recommended["price_unit"] == price_unit\n                        and recommended["price_inr"] <= max_price\n                    ) else "CHECK",\n                    "PASS" if recommended["available_quantity"] >= quantity else "CHECK",\n                    "PASS" if recommended["effective_delivery_days"] <= max_delivery_days else "CHECK",\n                    "PASS" if recommended["rating"] >= min_rating else "CHECK",\n                    "PASS" if recommended["warranty_months"] >= min_warranty_years * 12 else "CHECK"\n                ]\n            })\n\n            st.dataframe(\n                fit_data,\n                hide_index=True,\n                width="stretch"\n            )\n\n            st.caption(\n                "Status verifies the recommended supplier against each requirement "\n                "using the current input values."\n            )\n\n\n            # -----------------------------------------------\n            # EXPLANATION\n            # -----------------------------------------------\n\n            st.subheader("Why This Supplier?")\n\n            explanation_text = result["explanation"]\n\n            # Add line breaks to the flattened explanation text only.\n            for section in (\n                "🏆 RECOMMENDATION", "✅ REQUIREMENT FIT",\n                "💪 STRENGTHS", "⚖️ TRADE-OFFS"\n            ):\n                explanation_text = explanation_text.replace(section, "\\n" + section + "\\n")\n\n            for field_label in (\n                "Product:", "Price:", "Available stock:",\n                "Effective delivery:", "Rating:", "Warranty:"\n            ):\n                explanation_text = explanation_text.replace(field_label, "\\n" + field_label)\n\n            explanation_text = explanation_text.replace("• ", "\\n• ").strip()\n            st.markdown(explanation_text.replace("\\n", "  \\n"))\n\n\n            # -----------------------------------------------\n            # RISK FLAGS\n            # -----------------------------------------------\n\n            st.subheader("Supplier Risk Assessment")\n\n            risks = recommended["risk_flags"]\n\n            if risks:\n                st.warning(\n                    f"{len(risks)} potential risk flag(s) detected. "\n                    "Review these before making a procurement decision."\n                )\n\n                for risk in risks:\n                    st.write(f"⚠️ {risk}")\n\n            else:\n                st.success(\n                    "No major risk flags detected against the current thresholds."\n                )\n\n            st.caption(\n                "These flags are based on the available supplier data and "\n                "configured thresholds; they do not guarantee supplier performance."\n            )\n\n\n            # -----------------------------------------------\n            # RANKED SUPPLIER TABLE\n            # -----------------------------------------------\n\n            st.subheader("Ranked Supplier Shortlist")\n\n            ml_col1, ml_col2 = st.columns([1, 3])\n            with ml_col1:\n                st.metric(\n                    "Relative ML anomaly score",\n                    f"{float(recommended[\'ml_anomaly_score\']):.1f}/100",\n                    help="Isolation Forest percentile within this synthetic dataset. It is not a probability of failure."\n                )\n            with ml_col2:\n                st.write(f"**ML profile screening: {recommended[\'ml_risk_level\']}**")\n                st.caption(str(recommended.get("ml_risk_note", "")))\n\n            st.caption(\n                "Suppliers that passed all hard constraints, ranked by "\n                "your selected price, delivery, quality, and reliability priorities."\n            )\n\n            display_df = ranked_df.head(5).copy()\n\n            display_df = display_df[\n                [\n                    "rank",\n                    "supplier_name",\n                    "state",\n                    "price_inr",\n                    "price_unit",\n                    "available_quantity",\n                    "effective_delivery_days",\n                    "rating",\n                    "warranty_months",\n                    "overall_score",\n                    "ml_risk_level",\n                    "ml_anomaly_score"\n                ]\n            ]\n\n            display_df["Price"] = display_df.apply(\n                lambda row: (\n                    f"₹{row[\'price_inr\']:,.2f} {row[\'price_unit\']}"\n                ),\n                axis=1\n            )\n\n            display_df["Delivery (days)"] = (\n                display_df["effective_delivery_days"].astype(int)\n            )\n\n            display_df["Rating"] = display_df["rating"].map(\n                lambda value: f"{value:.2f}/5"\n            )\n\n            display_df["Warranty"] = (\n                display_df["warranty_months"].astype(int).astype(str)\n                + " months"\n            )\n\n            if "ml_risk_level" in display_df.columns:\n                display_df["ML profile risk"] = display_df["ml_risk_level"]\n                display_df["Anomaly score"] = display_df["ml_anomaly_score"].round(1)\n            display_df["Score"] = display_df["overall_score"].map(\n                lambda value: f"{value:.4f}"\n            )\n\n            display_df["Available Stock"] = (\n                display_df["available_quantity"].astype(int)\n            )\n\n            display_df = display_df[\n                [\n                    "rank",\n                    "supplier_name",\n                    "state",\n                    "Price",\n                    "Available Stock",\n                    "Delivery (days)",\n                    "Rating",\n                    "Warranty",\n                    "Score"\n                ]\n            ]\n\n            display_df.columns = [\n                "Rank",\n                "Supplier",\n                "Location",\n                "Price",\n                "Available Stock",\n                "Delivery (days)",\n                "Rating",\n                "Warranty",\n                "Score"\n            ]\n\n            st.dataframe(\n                display_df,\n                hide_index=True,\n                width="stretch"\n            )\n\n            st.caption(\n                "Scores are relative ranking scores, not percentages or "\n                "probabilities of supplier performance."\n            )\n\n\n            # -----------------------------------------------\n            # REQUEST FOR QUOTATION (RFQ) DRAFT\n            # -----------------------------------------------\n\n            st.divider()\n            st.subheader("Prepare a Quotation Request")\n\n            st.caption(\n                "Prepare a draft for the selected supplier. "\n                "This prototype does not send a real enquiry."\n            )\n\n            rfq_suppliers = ranked_df.head(5).copy()\n\n            if not rfq_suppliers.empty:\n                rfq_options = {\n                    (\n                        f"{row[\'supplier_name\']} | {row[\'state\']} | "\n                        f"₹{row[\'price_inr\']:,.2f} {row[\'price_unit\']}"\n                    ): idx\n                    for idx, row in rfq_suppliers.iterrows()\n                }\n\n                with st.form("rfq_draft_form"):\n                    selected_label = st.selectbox(\n                        "Select supplier",\n                        options=list(rfq_options.keys())\n                    )\n\n                    rfq_quantity = st.number_input(\n                        "Required quantity",\n                        min_value=1,\n                        value=int(quantity),\n                        step=1\n                    )\n\n                    rfq_destination = st.text_input(\n                        "Delivery destination",\n                        value=str(destination)\n                    )\n\n                    rfq_target_price = st.number_input(\n                        "Target price per unit (₹)",\n                        min_value=0.0,\n                        value=float(max_price),\n                        step=1.0\n                    )\n\n                    rfq_delivery_days = st.number_input(\n                        "Maximum acceptable delivery (days)",\n                        min_value=1,\n                        value=int(max_delivery_days),\n                        step=1\n                    )\n\n                    rfq_message = st.text_area(\n                        "Additional requirements",\n                        placeholder=(\n                            "Mention packaging, documentation, "\n                            "dispatch date or other requirements."\n                        )\n                    )\n\n                    create_rfq = st.form_submit_button(\n                        "Generate RFQ Draft",\n                        use_container_width=True\n                    )\n\n                if create_rfq:\n                    selected_idx = rfq_options[selected_label]\n                    selected_supplier = rfq_suppliers.loc[selected_idx]\n\n                    rfq_text = f"""REQUEST FOR QUOTATION\n\nSupplier: {selected_supplier[\'supplier_name\']}\nSupplier location: {selected_supplier[\'state\']}\n\nProduct: {variant}\nRequired quantity: {int(rfq_quantity)} {price_unit}\nDelivery destination: {rfq_destination}\nTarget price: ₹{rfq_target_price:,.2f} {price_unit}\nRequired delivery: Within {int(rfq_delivery_days)} days\n\nSupplier\'s listed price: ₹{selected_supplier[\'price_inr\']:,.2f} {selected_supplier[\'price_unit\']}\nEstimated delivery: {int(selected_supplier[\'effective_delivery_days\'])} days\n\nAdditional requirements:\n{rfq_message.strip() or "None specified"}\n\nPlease confirm your current price, stock availability, dispatch date,\ndelivery timeline and applicable taxes/transport charges.\n\nNote: This is a draft generated by a synthetic-data prototype.\nNo enquiry has been sent to the supplier.\n"""\n\n                    st.success("RFQ draft generated.")\n                    st.text_area(\n                        "Review and copy your RFQ",\n                        value=rfq_text,\n                        height=300\n                    )\n                    st.download_button(\n                        "Download RFQ as TXT",\n                        data=rfq_text,\n                        file_name="supplier_rfq_draft.txt",\n                        mime="text/plain"\n                    )\n\n\n        # ====================================================\n        # NO PERFECT MATCH\n        # ====================================================\n\n        else:\n\n            st.divider()\n\n            st.header("3. No Perfect Match Found")\n\n            st.warning(\n                "No supplier satisfies all hard constraints simultaneously."\n            )\n\n            near_matches = result["near_matches"]\n\n            if not near_matches.empty:\n\n                st.subheader("Closest Alternatives")\n\n                st.caption(\n                    "Suppliers are ranked first by the number of failed constraints, "\n                    "then by the severity of the gaps."\n                )\n\n                for i, (_, row) in enumerate(\n                    near_matches.head(5).iterrows(),\n                    start=1\n                ):\n\n                    with st.expander(\n                        f"{i}. {row[\'supplier_name\']}"\n                    ):\n\n                        c1, c2, c3, c4 = st.columns(4)\n\n                        with c1:\n                            st.write(\n                                f"**Price:** "\n                                f"₹{row[\'price_inr\']:,.2f} "\n                                f"{row[\'price_unit\']}"\n                            )\n\n                        with c2:\n                            st.write(\n                                f"**Seller Location:** "\n                                f"{row[\'state\']}"\n                            )\n\n                        with c3:\n                            st.write(\n                                f"**Available Stock:** "\n                                f"{int(row[\'available_quantity\']):,}"\n                            )\n\n                        with c4:\n                            st.write(\n                                f"**Delivery:** "\n                                f"{int(row[\'effective_delivery_days\'])} days"\n                            )\n\n                        c5, c6, c7 = st.columns(3)\n\n                        with c5:\n                            st.write(\n                                f"**Rating:** "\n                                f"{row[\'rating\']:.2f}/5"\n                            )\n\n                        with c6:\n                            st.write(\n                                f"**Warranty:** "\n                                f"{int(row[\'warranty_months\'])} months"\n                            )\n\n                        with c7:\n                            st.write(\n                                f"**Failed Constraints:** "\n                                f"{int(row[\'total_failures\'])}"\n                            )\n\n                        if "ml_risk_level" in row.index:\n                            st.write(f"**ML profile screening:** {row[\'ml_risk_level\']} (relative anomaly score: {float(row[\'ml_anomaly_score\']):.1f}/100)")\n                            st.caption(str(row.get("ml_risk_note", "")))\n                        st.write(\n                            "**Why it is not a perfect match:**"\n                        )\n\n                        for reason in row["failure_reasons"]:\n                            st.write(f"• {reason}")\n\n            else:\n\n                st.error(\n                    "No suitable near-match suppliers were found."\n                )\n\n\n# ============================================================\n# FOOTER\n# ============================================================\n\nst.divider()\n\nst.caption(\n    "AI B2B Supplier Selection Copilot | Synthetic procurement dataset | "\n    "Explainable supplier ranking"\n)'
llm_source = '"""LLM-assisted extraction of natural-language procurement requirements.\n\nThe LLM converts buyer language to structured fields. Deterministic validation and\ncopilot_engine remain responsible for hard constraints and supplier ranking.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom typing import Any\n\n\nFIELD_NAMES = (\n    "variant", "quantity", "location", "max_price", "price_unit",\n    "max_delivery_days", "min_rating", "min_warranty_years",\n    "price_priority", "delivery_priority", "quality_priority", "reliability_priority",\n)\n\n\ndef parse_procurement_request(\n    query: str,\n    *,\n    valid_variants: list[str],\n    valid_price_units: list[str],\n    valid_locations: list[str],\n    api_key: str | None = None,\n    model: str | None = None,\n) -> dict[str, Any]:\n    """Parse a buyer\'s free-text request into validated, optional form values.\n\n    Missing/ambiguous values remain None; the LLM is explicitly told not to guess.\n    The function sends only the request and allowed option lists, not supplier data.\n    """\n    query = (query or "").strip()\n    if not query:\n        raise ValueError("Enter a procurement request in plain language first.")\n\n    key = api_key or os.getenv("OPENAI_API_KEY")\n    if not key:\n        raise RuntimeError(\n            "OpenAI API key is missing. Enter it in the app sidebar or set OPENAI_API_KEY."\n        )\n\n    try:\n        from openai import OpenAI\n    except ImportError as exc:\n        raise RuntimeError("The openai package is missing. Install it with: pip install openai") from exc\n\n    allowed = {\n        "variant": valid_variants,\n        "price_unit": valid_price_units,\n        "location": valid_locations,\n    }\n    schema = {\n        "variant": "string|null (must exactly match one allowed variant)",\n        "quantity": "integer|null (positive units)",\n        "location": "string|null (must exactly match an allowed Indian state/territory)",\n        "max_price": "number|null (maximum unit price in INR)",\n        "price_unit": "string|null (must exactly match an allowed price unit)",\n        "max_delivery_days": "integer|null (maximum total delivery days)",\n        "min_rating": "number|null (0 to 5)",\n        "min_warranty_years": "number|null (minimum warranty years)",\n        "price_priority": "integer|null (0 to 100)",\n        "delivery_priority": "integer|null (0 to 100)",\n        "quality_priority": "integer|null (0 to 100)",\n        "reliability_priority": "integer|null (0 to 100)",\n        "clarifying_questions": "array of short strings for important missing or ambiguous details",\n        "interpreted_summary": "one short sentence summarising only what the buyer said",\n    }\n    instructions = f"""You are a procurement intake assistant for a B2B supplier selection tool.\nConvert the buyer\'s natural-language request into JSON only. Do not select suppliers,\nclaim supplier facts, or invent missing requirements. Use null for every field that is\nnot explicitly stated or unambiguously implied. If product wording does not map to one\nexact allowed variant, set variant=null and ask a clarifying question. If a location or\nprice unit is not an exact allowed value, set it null and ask. Prices are INR unit prices;\nnever infer a currency conversion. Rating is on a 0-5 scale. Priority weights are 0-100.\nIf the buyer says e.g. "delivery matters most", you may set delivery_priority=80 and\nother stated relative priorities accordingly, but leave all priorities null if no preference\nwas expressed. Return all schema fields, using null for unknown values.\n\nAllowed variants: {json.dumps(valid_variants, ensure_ascii=False)}\nAllowed price units: {json.dumps(valid_price_units, ensure_ascii=False)}\nAllowed locations: {json.dumps(valid_locations, ensure_ascii=False)}\nField meanings: {json.dumps(schema)}\n"""\n    client = OpenAI(api_key=key)\n    response = client.chat.completions.create(\n        model=model or os.getenv("OPENAI_MODEL", "gpt-4o-mini"),\n        temperature=0,\n        response_format={"type": "json_object"},\n        messages=[\n            {"role": "system", "content": instructions},\n            {"role": "user", "content": query[:6000]},\n        ],\n    )\n    content = response.choices[0].message.content or "{}"\n    try:\n        parsed = json.loads(content)\n    except json.JSONDecodeError as exc:\n        raise RuntimeError("The LLM returned invalid JSON. Please retry or simplify the request.") from exc\n\n    result: dict[str, Any] = {name: None for name in FIELD_NAMES}\n    for name in FIELD_NAMES:\n        value = parsed.get(name)\n        if value is None:\n            continue\n        if name == "variant":\n            if isinstance(value, str) and value in valid_variants:\n                result[name] = value\n        elif name == "price_unit":\n            if isinstance(value, str) and value in valid_price_units:\n                result[name] = value\n        elif name == "location":\n            if isinstance(value, str) and value in valid_locations:\n                result[name] = value\n        elif name in {"quantity", "max_delivery_days", "price_priority", "delivery_priority", "quality_priority", "reliability_priority"}:\n            try:\n                number = int(value)\n            except (TypeError, ValueError):\n                continue\n            if name in {"quantity", "max_delivery_days"} and number >= 1:\n                result[name] = number\n            elif name.endswith("_priority") and 0 <= number <= 100:\n                result[name] = number\n        else:\n            try:\n                number = float(value)\n            except (TypeError, ValueError):\n                continue\n            if name == "max_price" and number >= 0:\n                result[name] = number\n            elif name == "min_rating" and 0 <= number <= 5:\n                result[name] = number\n            elif name == "min_warranty_years" and number >= 0:\n                result[name] = number\n\n    questions = parsed.get("clarifying_questions", [])\n    result["clarifying_questions"] = [str(q)[:240] for q in questions[:5]] if isinstance(questions, list) else []\n    result["interpreted_summary"] = str(parsed.get("interpreted_summary", "")).strip()[:500]\n    # If the LLM tried to return a disallowed exact-choice value, make the ambiguity visible.\n    for name in ("variant", "price_unit", "location"):\n        raw_value = parsed.get(name)\n        if raw_value is not None and result[name] is None:\n            result["clarifying_questions"].append(f"Please confirm the {name.replace(\'_\', \' \')} from the available options.")\n    result["clarifying_questions"] = list(dict.fromkeys(result["clarifying_questions"]))[:5]\n    return result\n'
Path("app.py").write_text(app_source, encoding="utf-8")
Path("llm_procurement_assistant.py").write_text(llm_source, encoding="utf-8")
print("Wrote app.py and llm_procurement_assistant.py")


Wrote app.py and llm_procurement_assistant.py


## 5. Smoke tests

In [16]:
import pandas as pd, importlib, sys
sys.path.insert(0, str(Path.cwd()))
import copilot_engine
importlib.reload(copilot_engine)
listing_df = pd.read_csv("supplier_listings.csv")
supplier_df = pd.read_csv("supplier_master.csv")
requirement = {"variant":"Industrial LED Bulb 30W", "quantity":500, "location":"Maharashtra", "max_price":350.0, "price_unit":"per piece", "max_delivery_days":7, "min_rating":4.5, "min_warranty_years":2.0}
result = copilot_engine.run_supplier_copilot_dynamic(listing_df=listing_df, supplier_df=supplier_df, requirement=requirement, price_priority=20, delivery_priority=50, quality_priority=20, reliability_priority=10)
assert result["status"] in {"MATCH_FOUND", "NO_PERFECT_MATCH"}
print("Status:", result["status"])
print("Listings:", len(listing_df), "| Suppliers:", supplier_df["supplier_id"].nunique())
print("Ranked rows:", len(result.get("ranked_suppliers", [])))
print("Near-match rows:", len(result.get("near_matches", [])))
print("Smoke test passed.")
# Test the no-perfect-match path and its explanation.
strict_requirement = {"variant": "Industrial LED Bulb 30W", "quantity": 100000, "location": "Maharashtra", "max_price": 1.0, "price_unit": "per piece", "max_delivery_days": 1, "min_rating": 5.0, "min_warranty_years": 10.0}
strict_result = copilot_engine.run_supplier_copilot_dynamic(listing_df=listing_df, supplier_df=supplier_df, requirement=strict_requirement, price_priority=25, delivery_priority=25, quality_priority=25, reliability_priority=25)
assert strict_result["status"] == "NO_PERFECT_MATCH"
assert isinstance(strict_result.get("explanation"), str) and strict_result["explanation"].strip()
if not strict_result["near_matches"].empty:
    assert "failure_reasons" in strict_result["near_matches"].columns
print("No-perfect-match explanation test passed.")

assert "ml_anomaly_score" in result["ranked_suppliers"].columns if result["status"] == "MATCH_FOUND" else "ml_anomaly_score" in result["near_matches"].columns
print("ML anomaly-screening integration test passed.")


Status: MATCH_FOUND
Listings: 1020 | Suppliers: 200
Ranked rows: 4
Near-match rows: 0
Smoke test passed.
No-perfect-match explanation test passed.
ML anomaly-screening integration test passed.


## 6. Launch the app in Google Colab

In [17]:
# Run this in a Colab code cell, then open the served port iframe below.
!streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.enableCORS=false --server.enableXsrfProtection=false --server.headless=true &>/content/streamlit.log &

In [18]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>